# Занятие 8. Системная оптимизация: выбор экономичных моделей и архитектурная реструктуризация

На прошлом занятии мы уменьшали контекст одного RAG-вызова. Здесь задача меняется: нужные данные живут в нескольких системах, поэтому одного заранее собранного запроса уже не хватает.

Внутри занятия мы:

- найдём границу конфигурации `E` на вопросах о сотрудниках, командах и задачах;
- опишем обычные Python-функции так, чтобы модель могла ими пользоваться;
- соберём SGR next-step и ReAct поверх одних данных и инструментов;
- сравним ответ, обязательные факты, вызванные функции, шаги, токены и задержку;
- разделим полную траекторию запуска и рабочий контекст следующего вызова;
- проверим, приносит ли отдельный планировщик пользу на многошаговых задачах.

> Ноутбук рассчитан на Google Colaboratory, локальный запуск тоже поддержан. Модельные ячейки обращаются к GigaChat, поэтому нужен секрет `API_KEY_GIGA` или `GIGACHAT_CREDENTIALS`. Проверка TLS в учебной конфигурации выключена по умолчанию. Для явного включения задайте `GIGACHAT_VERIFY_SSL=1`.


## Введение

Конфигурация `E` неплохо живёт, пока ответ лежит в корпоративных политиках. Поиск приносит несколько подходящих фрагментов, модель читает их и формирует ответ. Один вопрос, один RAG, один вызов.

Потом приходит вполне обычный продуктовый запрос:

> А бот ответит мне "Кто руководит командой Анны Петровой и сколько открытых задач по оценке агентов назначено участникам этой команды?"

Здесь уже три источника. Анну нужно найти в справочнике сотрудников, состав и руководителя получить из справочника команд, задачи посчитать в трекере.

Можно заранее положить в промпт все таблицы. На двенадцати учебных задачах это даже сработает. В реальной системе такой запрос быстро превращается в прикол: модель каждый раз тащит сотрудников, команды и задачи, хотя ей нужны одна строка и одно число.

Поэтому дальше приложение будет подгружать данные по ходу решения. А как – сейчас обсудим.

### Маршрут занятия

- Вернём конфигурацию `E` и покажем вопрос, на который ей нечем отвечать.
- Добавим синтетические справочники сотрудников, команд и задач.
- Напишем функции доступа к данным и отдельные схемы для модели.
- Зафиксируем 18 простых проверок до того, как начнём собирать циклы.
- Реализуем SGR next-step и посмотрим его траекторию.
- Проведём один вызов функции с возвратом результата в GigaChat.
- Повторим этот обмен в ReAct-цикле и увидим, как растёт история.
- Добавим пять многошаговых кейсов, сравним архитектуры и проверим планировщик.

Все числа ниже появляются из выполненных ячеек. Подготовленных ответов модели в ноутбуке нет. По умолчанию запускаются `T03`, `S03` и `M02`. Переменная `LESSON8_FULL_BENCHMARK=1` включает полный набор из 23 заданий.


## Установка зависимостей

Набор библиотек небольшой:

- `gigachat==0.2.3` даёт структурированный вывод и нативные вызовы функций;
- `pydantic` проверяет схемы действий и аргументов;
- `pandas` и `matplotlib` показывают результаты прогонов;
- `python-dotenv` читает локальные секреты.

В Colab недостающие пакеты устанавливаются автоматически. При локальном запуске ячейка останавливается и печатает готовую команду установки. Это полезнее тихой попытки продолжить работу с другой версией SDK, а потом полчаса искать, почему у ответа внезапно другая структура.


In [ ]:
import importlib.util
import subprocess
import sys
from importlib.metadata import version as distribution_version

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False

required_modules = {
    "gigachat": "gigachat==0.2.3",
    "dotenv": "python-dotenv>=1.0,<2",
    "pydantic": "pydantic>=2.7,<3",
    "pandas": "pandas>=2.2,<3",
    "matplotlib": "matplotlib>=3.8,<4",
    "ipywidgets": "ipywidgets>=8.1,<9",
}

packages_to_install = [
    package
    for module, package in required_modules.items()
    if importlib.util.find_spec(module) is None
]

installed_gigachat_version = (
    distribution_version("gigachat")
    if importlib.util.find_spec("gigachat") is not None
    else None
)
if installed_gigachat_version not in {None, "0.2.3"}:
    packages_to_install.append(required_modules["gigachat"])

if packages_to_install and IN_COLAB:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-qU", *packages_to_install]
    )
elif packages_to_install:
    install_command = "python -m pip install " + " ".join(
        f'"{package}"' for package in packages_to_install
    )
    raise RuntimeError(
        f"Установите зависимости и перезапустите ядро:\n{install_command}"
    )
else:
    print("Зависимости и версия GigaChat SDK проверены.")

In [ ]:
from __future__ import annotations

import copy
import json
import os
import re
import sys
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Callable, Literal, Optional, Type, Union

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from gigachat import GigaChat
from IPython.display import Markdown, display
from pydantic import BaseModel, ConfigDict, Field

pd.set_option("display.max_colwidth", 120)


def show_table(frame: pd.DataFrame, *, rows: Optional[int] = None) -> None:
    prepared = frame.head(rows) if rows is not None else frame
    display(prepared)


print("GigaChat SDK доступен.")

### Подключение к GigaChat

Секрет ищется сначала в `API_KEY_GIGA`, затем в `GIGACHAT_CREDENTIALS`. Если ключа нет, выполнение останавливается сразу.

Короткий режим запускает только несколько сквозных кейсов. Полный прогон включается так:

```bash
export LESSON8_FULL_BENCHMARK=1
```

Проверка TLS в этом ноутбуке выключена ради совместимости с учебным окружением. Для продакшена такое значение лучше не переносить.


In [ ]:
load_dotenv(Path.cwd() / ".env", override=False)


def load_secret(name: str) -> str:
    value = os.getenv(name, "").strip()
    if value:
        return value

    if IN_COLAB:
        from google.colab import userdata

        return (userdata.get(name) or "").strip()
    return ""


API_KEY_GIGA = load_secret("API_KEY_GIGA") or load_secret("GIGACHAT_CREDENTIALS")
MODEL_NAME = os.getenv("GIGACHAT_MODEL", "GigaChat-2-Max")
SCOPE = os.getenv("GIGACHAT_SCOPE", "GIGACHAT_API_B2B")
BASE_URL = os.getenv("GIGACHAT_BASE_URL", "https://api.giga.chat/v1")
VERIFY_SSL_CERTS = os.getenv("GIGACHAT_VERIFY_SSL", "0").strip() == "1"
CA_BUNDLE_FILE = os.getenv("GIGACHAT_CA_BUNDLE_FILE") or None
FULL_BENCHMARK = os.getenv("LESSON8_FULL_BENCHMARK", "0") == "1"

if not API_KEY_GIGA:
    raise RuntimeError(
        "Секрет GigaChat не найден в API_KEY_GIGA или GIGACHAT_CREDENTIALS."
    )

client = GigaChat(
    base_url=BASE_URL,
    credentials=API_KEY_GIGA,
    scope=SCOPE,
    model=MODEL_NAME,
    verify_ssl_certs=VERIFY_SSL_CERTS,
    ca_bundle_file=CA_BUNDLE_FILE,
)


print("Модель генерации:", MODEL_NAME)
print("Тип доступа:", SCOPE)
print("Проверка TLS:", VERIFY_SSL_CERTS)
print("Ключ загружен:", API_KEY_GIGA not in {""})
print("Запуск полного бенчмарка:", FULL_BENCHMARK)

## Возвращаем конфигурацию E

Сначала восстановим исходную точку. Иначе после добавления функций будет трудно понять, какую проблему мы вообще решили.

Конфигурация `E` получает заранее найденные фрагменты политик и делает один генеративный вызов. Справочника сотрудников, состава команд и данных трекера в ней нет.

| Конфигурация | Контекст | Инструкция | Пример | Схема | `max_tokens` | Температура |
|---|---|---|---|---|---:|---:|
| `E` | компактный | компактная | один | компактная | 240 | 0,01 |

Теперь зададим ей вопрос, для которого текстовых политик недостаточно.


In [ ]:
@dataclass(frozen=True)
class PipelineConfig:
    config_id: str
    name: str
    context_mode: str
    prompt_mode: str
    example_mode: str
    schema_mode: str
    max_tokens: int
    temperature: float


CONFIG_E = PipelineConfig(
    config_id="E",
    name="E: компактная схема",
    context_mode="compact",
    prompt_mode="compact",
    example_mode="one",
    schema_mode="compact",
    max_tokens=240,
    temperature=0.01,
)

assert CONFIG_E.config_id == "E"
assert CONFIG_E.max_tokens == 240

### Смотрим на пробел в данных

У старого пайплайна контекст собирается до вызова модели. Поиск работает только по корпусу корпоративных политик.

Отправим новый вопрос в GigaChat через конфигурацию `E`. В контекст попадут только два фрагмента корпоративных политик. Функций и справочников у модели пока нет.


In [ ]:
NEW_PRODUCT_QUESTION = (
    "Кто руководит командой Анны Петровой и сколько открытых задач "
    "по оценке агентов назначено участникам этой команды?"
)

CONFIG_E_POLICY_CONTEXT = """
[Лимит переноса отпуска]
На следующий календарный год можно перенести не более пяти неиспользованных дней ежегодного оплачиваемого отпуска.

[Работа из другой страны]
До начала работы из другой страны нужны письменные согласования HR, юридической службы и службы информационной безопасности.
""".strip()

config_e_request = {
    "messages": [
        {
            "role": "system",
            "content": [
                {
                    "text": (
                        "Отвечай только по переданным фрагментам корпоративных политик. "
                        "Если в них нет нужных данных, прямо скажи, какого источника не хватает. "
                        "Не угадывай имена и числа."
                    )
                }
            ],
        },
        {
            "role": "user",
            "content": [
                {
                    "text": f"Контекст:\n{CONFIG_E_POLICY_CONTEXT}\n\nВопрос:\n{NEW_PRODUCT_QUESTION}"
                }
            ],
        },
    ],
    "model_options": {
        "temperature": CONFIG_E.temperature,
        "max_tokens": CONFIG_E.max_tokens,
    },
}

config_e_response = client.chat.create(config_e_request)
config_e_payload = (
    config_e_response.model_dump(mode="json", exclude_none=True, by_alias=True)
    if hasattr(config_e_response, "model_dump")
    else dict(config_e_response)
)
config_e_answer = ""
for message in config_e_payload.get("messages", []):
    content = message.get("content") or []
    if isinstance(content, str):
        config_e_answer = content
        break
    text_parts = [
        str(part["text"])
        for part in content
        if isinstance(part, dict) and part.get("text")
    ]
    if text_parts:
        config_e_answer = "".join(text_parts)
        break

if not config_e_answer:
    raise RuntimeError("GigaChat не вернул текстовый ответ для конфигурации E.")

print("Вопрос:", NEW_PRODUCT_QUESTION)
print("Реальный ответ конфигурации E:")
print(config_e_answer)

Это уже настоящий ответ модели. GigaChat запросил два источника: организационную структуру с руководителями команд и систему управления задачами. Конкретного имени и числа в ответе нет. Значит, конфигурация `E` честно дошла до границы своих данных и не стала сочинять ответ из воздуха. Это вообще-то круто!

Передавать полные таблицы в каждый запрос незачем. Для этого вопроса нужны карточка Анны, одна команда и отфильтрованное число задач. Дальше дадим приложению небольшие функции доступа к этим источникам.


## Добавляем сотрудников, команды и задачи

Данные ниже полностью синтетические. Они небольшие, но разложены по тем же границам, которые встречаются в корпоративном помощнике:

- `POLICIES` хранит текстовые правила;
- `EMPLOYEES` и `TEAMS` образуют справочник людей и команд;
- `TASKS` имитирует трекер с проектами, статусами, исполнителями и метками.

Пока всё лежит в памяти Python. Позже в контекст модели попадёт только результат выбранной функции. Благодаря этому в траектории можно будет увидеть, откуда взялось конкретное имя, `team_id` или число задач.

| `team_id` | Команда | Руководитель | Участников |
|---|---|---|---:|
| `team-agent-platform` | Agent Platform | Илья Морозов | 4 |
| `team-data-quality` | Data Quality | Ольга Лебедева | 3 |
| `team-support-operations` | Support Operations | Екатерина Волкова | 3 |

| Задача | Проект | Статус | Исполнитель | Метки |
|---|---|---|---|---|
| `TASK-101` | Support Automation | `in_progress` | `E002` | `eval`, `agents` |
| `TASK-102` | Support Automation | `open` | `E003` | `eval`, `rag` |
| `TASK-103` | Support Automation | `blocked` | `E004` | `tools`, `prompt` |
| `TASK-104` | Support Automation | `open` | `E002` | `eval`, `agents` |
| `TASK-105` | Support Automation | `done` | `E004` | `context`, `prompt` |
| `TASK-106` | Support Automation | `in_progress` | `E003` | `eval`, `judge` |
| `TASK-107` | Support Automation | `done` | `E009` | `docs` |
| `TASK-108` | Support Automation | `open` | `E004` | `tools`, `prompt` |
| `TASK-109` | Data Quality | `open` | `E006` | `analytics` |
| `TASK-110` | Data Quality | `blocked` | `E007` | `eval`, `clustering` |
| `TASK-111` | Support Operations | `in_progress` | `E009` | `operations` |
| `TASK-112` | Support Operations | `open` | `E010` | `prompt` |


In [ ]:
@dataclass(frozen=True)
class PolicyChunk:
    chunk_id: str
    title: str
    text: str
    keywords: tuple[str, ...]


@dataclass(frozen=True)
class Employee:
    employee_id: str
    name: str
    email: str
    team_id: str
    role: str


@dataclass(frozen=True)
class Team:
    team_id: str
    name: str
    lead_employee_id: str
    member_ids: tuple[str, ...]


@dataclass(frozen=True)
class Task:
    task_id: str
    title: str
    project: str
    status: Literal["open", "in_progress", "blocked", "done"]
    assignee_id: str
    labels: tuple[str, ...]


POLICIES = {
    "vacation#carryover": PolicyChunk(
        "vacation#carryover",
        "Лимит переноса отпуска",
        "На следующий календарный год можно перенести не более пяти неиспользованных дней ежегодного оплачиваемого отпуска.",
        ("отпуск", "перенос", "перенести", "дней", "пять", "5", "шесть", "6"),
    ),
    "vacation#approval": PolicyChunk(
        "vacation#approval",
        "Согласование переноса",
        "Перенос требует предварительного письменного согласования непосредственного руководителя.",
        ("отпуск", "перенос", "согласование", "письменное", "руководитель"),
    ),
    "sick#notification": PolicyChunk(
        "sick#notification",
        "Уведомление о болезни",
        "В первый день болезни сотрудник уведомляет непосредственного руководителя и HR.",
        ("болезнь", "болею", "больничный", "первый", "день", "руководитель", "hr"),
    ),
    "remote#abroad": PolicyChunk(
        "remote#abroad",
        "Работа из другой страны",
        "До начала работы из другой страны нужны письменные согласования HR, юридической службы и службы информационной безопасности.",
        (
            "другая",
            "страна",
            "зарубеж",
            "удаленная",
            "работа",
            "hr",
            "юридическая",
            "безопасность",
        ),
    ),
    "expenses#travel": PolicyChunk(
        "expenses#travel",
        "Командировочные расходы",
        "Командировочные расходы оформляются после возвращения.",
        ("командировка", "расходы", "возвращение"),
    ),
}

EMPLOYEES = {
    "E001": Employee(
        "E001",
        "Илья Морозов",
        "ilya.morozov@llm_gigachat_course.ru",
        "team-agent-platform",
        "ML Lead",
    ),
    "E002": Employee(
        "E002",
        "Анна Петрова",
        "anna.petrova@llm_gigachat_course.ru",
        "team-agent-platform",
        "ML Engineer",
    ),
    "E003": Employee(
        "E003",
        "Мария Соколова",
        "maria.sokolova@llm_gigachat_course.ru",
        "team-agent-platform",
        "Data Analyst",
    ),
    "E004": Employee(
        "E004",
        "Артем Волков",
        "artem.volkov@llm_gigachat_course.ru",
        "team-agent-platform",
        "Backend Engineer",
    ),
    "E005": Employee(
        "E005",
        "Ольга Лебедева",
        "olga.lebedeva@llm_gigachat_course.ru",
        "team-data-quality",
        "Data Quality Lead",
    ),
    "E006": Employee(
        "E006",
        "Никита Козлов",
        "nikita.kozlov@llm_gigachat_course.ru",
        "team-data-quality",
        "Data Engineer",
    ),
    "E007": Employee(
        "E007",
        "София Орлова",
        "sofia.orlova@llm_gigachat_course.ru",
        "team-data-quality",
        "Analyst",
    ),
    "E008": Employee(
        "E008",
        "Екатерина Волкова",
        "ekaterina.volkova@llm_gigachat_course.ru",
        "team-support-operations",
        "Support Operations Lead",
    ),
    "E009": Employee(
        "E009",
        "Дмитрий Попов",
        "dmitry.popov@llm_gigachat_course.ru",
        "team-support-operations",
        "Support Specialist",
    ),
    "E010": Employee(
        "E010",
        "Полина Смирнова",
        "polina.smirnova@llm_gigachat_course.ru",
        "team-support-operations",
        "Product Manager",
    ),
}

TEAMS = {
    "team-agent-platform": Team(
        "team-agent-platform",
        "Agent Platform",
        "E001",
        ("E001", "E002", "E003", "E004"),
    ),
    "team-data-quality": Team(
        "team-data-quality", "Data Quality", "E005", ("E005", "E006", "E007")
    ),
    "team-support-operations": Team(
        "team-support-operations",
        "Support Operations",
        "E008",
        ("E008", "E009", "E010"),
    ),
}

TASKS = [
    Task(
        "TASK-101",
        "Собрать golden set для оценки агентов",
        "Support Automation",
        "in_progress",
        "E002",
        ("eval", "agents"),
    ),
    Task(
        "TASK-102",
        "Добавить проверку фактичности",
        "Support Automation",
        "open",
        "E003",
        ("eval", "rag"),
    ),
    Task(
        "TASK-103",
        "Починить описание функции поиска задач",
        "Support Automation",
        "blocked",
        "E004",
        ("tools", "prompt"),
    ),
    Task(
        "TASK-104",
        "Проверить регрессии ReAct",
        "Support Automation",
        "open",
        "E002",
        ("eval", "agents"),
    ),
    Task(
        "TASK-105",
        "Сжать контекст после 8000 токенов",
        "Support Automation",
        "done",
        "E004",
        ("context", "prompt"),
    ),
    Task(
        "TASK-106",
        "Калибровать judge на human labels",
        "Support Automation",
        "in_progress",
        "E003",
        ("eval", "judge"),
    ),
    Task(
        "TASK-107",
        "Обновить onboarding команды",
        "Support Automation",
        "done",
        "E009",
        ("docs",),
    ),
    Task(
        "TASK-108",
        "Описать политику function calling",
        "Support Automation",
        "open",
        "E004",
        ("tools", "prompt"),
    ),
    Task(
        "TASK-109",
        "Собрать отчёт по обращениям",
        "Data Quality",
        "open",
        "E006",
        ("analytics",),
    ),
    Task(
        "TASK-110",
        "Проверить дрейф кластеров",
        "Data Quality",
        "blocked",
        "E007",
        ("eval", "clustering"),
    ),
    Task(
        "TASK-111",
        "Разобрать ручные эскалации",
        "Support Operations",
        "in_progress",
        "E009",
        ("operations",),
    ),
    Task(
        "TASK-112",
        "Обновить макрос ответа",
        "Support Operations",
        "open",
        "E010",
        ("prompt",),
    ),
]

OPEN_STATUSES = {"open", "in_progress", "blocked"}

assert all(team.lead_employee_id in team.member_ids for team in TEAMS.values())
assert all(employee.team_id in TEAMS for employee in EMPLOYEES.values())
assert all(task.assignee_id in EMPLOYEES for task in TASKS)

In [ ]:
assert len(EMPLOYEES) == 10
assert len(TEAMS) == 3
assert len(TASKS) == 12

## Делаем функции для доступа к данным

Под функцией здесь понимается обычный Python-функции с отдельным контрактом для модели. `find_employee` ищет сотрудника, `get_team` возвращает команду, `search_tasks` применяет фильтры трекера, `search_policy_constraints` работает с политиками.

Сначала вызовем эти функции напрямую. GigaChat подключим позже, когда убедимся, что сами функции возвращают ожидаемые данные.

У каждой функции есть две стороны:

- Python-код и JSON-результат, с которыми работает приложение;
- название, описание и JSON Schema, которые видит модель.

Вторая сторона обычно требует больше текста. Из сигнатуры `is_open: bool` модель не узнает, какие статусы считаются открытыми и можно ли совместить этот фильтр с `labels`.


In [ ]:
WORD_RE = re.compile(r"[a-zа-яё0-9-]+", flags=re.IGNORECASE)


def normalize_text(value: str) -> str:
    return " ".join(WORD_RE.findall(value.lower().replace("ё", "е")))


def search_policy_constraints(query: str, limit: int = 3) -> dict[str, Any]:
    """Возвращает релевантные ограничения из учебного корпуса политик."""
    query_tokens = set(normalize_text(query).split())
    ranked = []
    for chunk in POLICIES.values():
        keyword_tokens = {normalize_text(item) for item in chunk.keywords}
        text_tokens = set(normalize_text(chunk.text + " " + chunk.title).split())
        score = 2 * len(query_tokens & keyword_tokens) + len(query_tokens & text_tokens)
        if score > 0:
            ranked.append((score, chunk))

    ranked.sort(key=lambda item: (-item[0], item[1].chunk_id))
    items = [
        {
            "chunk_id": chunk.chunk_id,
            "title": chunk.title,
            "text": chunk.text,
            "score": score,
        }
        for score, chunk in ranked[: max(1, min(limit, 5))]
    ]
    return {"query": query, "items": items, "count": len(items)}


def find_employee(query: str) -> dict[str, Any]:
    """Ищет одного сотрудника по имени, почте или employee_id."""
    needle = normalize_text(query)
    matches = []
    for employee in EMPLOYEES.values():
        haystack = normalize_text(
            f"{employee.employee_id} {employee.name} {employee.email}"
        )
        if needle in haystack or all(part in haystack for part in needle.split()):
            team = TEAMS[employee.team_id]
            matches.append(
                {
                    "employee_id": employee.employee_id,
                    "name": employee.name,
                    "email": employee.email,
                    "role": employee.role,
                    "team_id": team.team_id,
                    "team_name": team.name,
                }
            )
    return {"query": query, "matches": matches, "count": len(matches)}


def get_team(team: str) -> dict[str, Any]:
    """Возвращает руководителя и участников одной команды."""
    needle = normalize_text(team)
    matches = [
        item
        for item in TEAMS.values()
        if needle in normalize_text(f"{item.team_id} {item.name}")
    ]
    prepared = []
    for item in matches:
        prepared.append(
            {
                "team_id": item.team_id,
                "name": item.name,
                "lead": asdict(EMPLOYEES[item.lead_employee_id]),
                "members": [
                    {"employee_id": member_id, "name": EMPLOYEES[member_id].name}
                    for member_id in item.member_ids
                ],
                "member_count": len(item.member_ids),
            }
        )
    return {"query": team, "matches": prepared, "count": len(prepared)}


def resolve_employee_ids(assignee: Optional[str]) -> Optional[set[str]]:
    if assignee is None:
        return None
    result = find_employee(assignee)
    return {item["employee_id"] for item in result["matches"]}


def search_tasks(
    query: Optional[str] = None,
    project: Optional[str] = None,
    status: Optional[str] = None,
    is_open: Optional[bool] = None,
    assignee: Optional[str] = None,
    labels: Optional[list[str]] = None,
    limit: int = 20,
) -> dict[str, Any]:
    """Фильтрует задачи и возвращает компактные строки плюс итоговое число."""
    assignee_ids = resolve_employee_ids(assignee)
    normalized_query = normalize_text(query or "")
    required_labels = {item.lower() for item in labels or []}

    result = []
    for task in TASKS:
        if normalized_query:
            task_haystack = normalize_text(f"{task.task_id} {task.title}")
            if not all(part in task_haystack for part in normalized_query.split()):
                continue
        if project and normalize_text(project) not in normalize_text(task.project):
            continue
        if status and task.status != status:
            continue
        if is_open is True and task.status not in OPEN_STATUSES:
            continue
        if is_open is False and task.status in OPEN_STATUSES:
            continue
        if assignee_ids is not None and task.assignee_id not in assignee_ids:
            continue
        if required_labels and not required_labels.issubset(set(task.labels)):
            continue

        employee = EMPLOYEES[task.assignee_id]
        result.append(
            {
                "task_id": task.task_id,
                "title": task.title,
                "project": task.project,
                "status": task.status,
                "is_open": task.status in OPEN_STATUSES,
                "assignee_id": employee.employee_id,
                "assignee_name": employee.name,
                "team_id": employee.team_id,
                "labels": list(task.labels),
            }
        )

    limited = result[: max(1, min(limit, 50))]
    return {
        "filters": {
            "query": query,
            "project": project,
            "status": status,
            "is_open": is_open,
            "assignee": assignee,
            "labels": labels,
        },
        "items": limited,
        "count": len(result),
        "truncated": len(result) > len(limited),
    }

In [ ]:
print(json.dumps(find_employee("Анна Петрова"), ensure_ascii=False, indent=2))
print("-" * 100)
print(json.dumps(get_team("Agent Platform"), ensure_ascii=False, indent=2)[:1200])
print("-" * 100)
print(
    json.dumps(
        search_tasks(project="Support Automation", is_open=True, labels=["eval"]),
        ensure_ascii=False,
        indent=2,
    )[:1800]
)

### Описываем функции для модели

Хорошая схема отвечает хотя бы на четыре вопроса:

- в какой ситуации нужна функция;
- какие сущности и фильтры она принимает;
- что означают поля результата;
- где проходят границы между похожими инструментами.

Самая неприятная ошибка обычно выглядит вполне разумно. Пользователь просит все незавершённые задачи, модель передаёт `status="open"`, а функция честно возвращает только один статус. У нас `is_open=true` включает `open`, `in_progress` и `blocked`, поэтому это определение записано прямо в схеме.

Ещё одна деталь: `count` содержит полное число совпадений, а `items` может быть обрезан параметром `limit`. Для ответа на вопрос о количестве нужна первая величина.

В статье Anthropic [Writing effective tools for AI agents](https://www.anthropic.com/engineering/writing-tools-for-agents) описание инструмента предлагают писать как инструкцию для нового инженера: раскрывать неочевидные термины, возвращать полезный контекст и проверять изменения на задачах и полных траекториях. Ниже мы будем делать именно это.

| Функция | Когда нужна | Важная часть результата |
|---|---|---|
| `search_policy_constraints` | правило или ограничение | короткие фрагменты с `chunk_id` |
| `find_employee` | сотрудник и его текущая команда | `employee_id`, роль, `team_id` |
| `get_team` | руководитель и состав команды | `lead`, `members`, `member_count` |
| `search_tasks` | задачи по фильтрам | `items`, полный `count`, `truncated` |


In [ ]:
TOOL_SPECS = [
    {
        "name": "search_policy_constraints",
        "description": (
            "Ищет правила и ограничения в корпоративных политиках. Используй для вопросов "
            "об отпуске, болезни, работе из другой страны и других нормативных условиях. "
            "Возвращает короткие фрагменты с chunk_id."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Смысловой поисковый запрос с ключевыми условиями пользователя.",
                },
                "limit": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 5,
                    "default": 3,
                    "description": "Максимальное число коротких фрагментов.",
                },
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    {
        "name": "find_employee",
        "description": (
            "Ищет сотрудника по полному или частичному имени, рабочей почте или employee_id. "
            "Возвращает роль и текущую команду. Не используй для поиска руководителя всей команды, "
            "если название команды уже известно."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Имя, рабочая почта или employee_id сотрудника.",
                }
            },
            "required": ["query"],
            "additionalProperties": False,
        },
    },
    {
        "name": "get_team",
        "description": (
            "Возвращает одну команду, её руководителя, участников и число участников. "
            "Используй, когда известно название или team_id команды."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "team": {
                    "type": "string",
                    "description": "Название команды или team_id.",
                }
            },
            "required": ["team"],
            "additionalProperties": False,
        },
    },
    {
        "name": "search_tasks",
        "description": (
            "Ищет задачи трекера по проекту, статусу, открытости, исполнителю, меткам и словам в названии. "
            "is_open=true включает open, in_progress и blocked. Поле count содержит число всех совпадений "
            "до ограничения limit. Используй count для точного числового ответа."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Слова из task_id или названия задачи. Не передавай поле, если фильтр не нужен.",
                },
                "project": {
                    "type": "string",
                    "description": "Название проекта. Не передавай поле, если проект неизвестен или не важен.",
                },
                "status": {
                    "type": "string",
                    "enum": ["open", "in_progress", "blocked", "done"],
                    "description": "Точный статус. Для всех незавершённых задач используй is_open=true.",
                },
                "is_open": {
                    "type": "boolean",
                    "description": "true для open, in_progress и blocked; false только для завершённых. Не передавай поле без фильтра.",
                },
                "assignee": {
                    "type": "string",
                    "description": "Имя, почта или employee_id одного исполнителя. Не передавай поле без фильтра.",
                },
                "labels": {
                    "type": "array",
                    "items": {"type": "string"},
                    "description": "Все перечисленные метки должны быть у задачи. Не передавай поле без фильтра.",
                },
                "limit": {
                    "type": "integer",
                    "minimum": 1,
                    "maximum": 50,
                    "default": 20,
                    "description": "Сколько строк вернуть. Поле count всё равно считает все совпадения.",
                },
            },
            "additionalProperties": False,
        },
    },
]

TOOL_HANDLERS: dict[str, Callable[..., dict[str, Any]]] = {
    "search_policy_constraints": search_policy_constraints,
    "find_employee": find_employee,
    "get_team": get_team,
    "search_tasks": search_tasks,
}

assert {item["name"] for item in TOOL_SPECS} == set(TOOL_HANDLERS)

### Сначала проверяем функции без модели

До первого LLM-вызова прогоняем обычные Python-проверки. Иначе при неправильном числе будет непонятно, кто ошибся: модель выбрала плохой фильтр или функция неверно посчитала данные.

Здесь проверяем состав `is_open`, поиск по имени, сохранение полного `count` при маленьком `limit` и отсутствие дубля руководителя в составе команды.


In [ ]:
assert find_employee("Анна Петрова")["matches"][0]["team_name"] == "Agent Platform"
assert get_team("Agent Platform")["matches"][0]["member_count"] == 4
assert (
    search_tasks(project="Support Automation", is_open=True, labels=["eval"])["count"]
    == 4
)
assert search_tasks(project="Support Automation", status="blocked")["count"] == 1
assert search_tasks(project="Support Automation", labels=["prompt"])["count"] == 3
assert search_tasks(query="TASK-108")["items"][0]["assignee_name"] == "Артем Волков"

print("Локальные контракты функций прошли проверку.")

### Почему название функции тоже часть архитектуры

Допустим, в системе есть функция `find(q, mode)`. Разработчик знает, что `q` ищет задачу по заголовку, а `mode="open"` означает три незавершённых статуса. Для модели это две короткие строки без привычного контекста. Она может выбрать неверный режим или придумать правдоподобную функцию вроде `search_open_tasks`, которой в схеме вообще нет.

Авторы [PA-Tool](https://arxiv.org/abs/2510.07248) называют такой класс ошибок несовпадением схемы. Их идея состоит в том, чтобы подстраивать имена компонентов под знакомые модели шаблоны без дообучения:

- модель несколько раз генерирует возможное имя по описанию компонента;
- для каждого кандидата считается `peakedness`, то есть число похожих вариантов рядом с ним по посимвольному расстоянию редактирования;
- выбирается кандидат из самой плотной группы.

На MetaTool и RoTBench авторы сообщают прирост до 17% и снижение ошибок несовпадения схемы на 80%. Эти числа относятся к их моделям, схемам и протоколу оценки. Для нашего трекера вывод скромнее: имена и параметры надо проверять на собственных кейсах, прежде чем компенсировать плохой интерфейс длинным промптом.

Поэтому здесь используются `find_employee`, `get_team` и `search_tasks`. По названию уже примерно понятно, какую сущность вернёт каждый инструмент.


#### Мини-практика: назовите функцию

Есть функция, который создаёт задачу в выбранном проекте и возвращает `task_id`.

Придумайте для него:

- название;
- короткое описание границ;
- ясные имена параметров;
- явный признак того, что вызов изменяет данные.

Ниже лежит один рабочий вариант. Он не подключён к агенту занятия, поэтому оценочные прогоны не меняют содержимое трекера.


In [ ]:
CREATE_TASK_EXAMPLE = {
    "name": "create_project_task",
    "description": (
        "Создаёт новую задачу в существующем проекте. Функция изменяет данные. "
        "Вызывай только после явной просьбы пользователя создать задачу. "
        "Возвращает task_id и сохранённые поля."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "project": {
                "type": "string",
                "description": "Точное название существующего проекта.",
            },
            "title": {
                "type": "string",
                "minLength": 5,
                "description": "Короткий заголовок действия.",
            },
            "assignee_employee_id": {
                "type": "string",
                "description": "employee_id исполнителя. Не передавай поле, если исполнитель не назначен.",
            },
            "labels": {
                "type": "array",
                "items": {"type": "string"},
                "description": "Метки задачи. Пустой список допустим.",
            },
        },
        "required": ["project", "title", "labels"],
        "additionalProperties": False,
    },
}

print(json.dumps(CREATE_TASK_EXAMPLE, ensure_ascii=False, indent=2))

## Расширяем оценочный набор до 18 кейсов

Две красивые траектории ещё не дают сравнения архитектур. До сборки циклов зафиксируем задачи и ожидаемые результаты.

Старые восемь вопросов из занятия 7 остаются регрессионным набором. После подключения инструментов агент всё ещё должен корректно отвечать по политикам.

К ним добавляются десять простых задач, каждая работает с одним новым источником или одной функцией. Для кейса заранее записаны:

- основной маркер `<yes>`, `<no>` или `<count:n>`;
- обязательные факты в итоговом ответе;
- функции, которые ожидаются в проверяемом маршруте.

Последний пункт пока грубый. Проверяется только наличие имени функции. Аргументы, лишние вызовы и допустимые альтернативные маршруты мы обсудим после первого прогона.

| Кейсы | Срез | Ожидаемый результат | Функция |
|---|---|---|---|
| `C01`, `T01`, `T02`, `T04`, `T06` | политики, отрицательные условия | `<no>` | `search_policy_constraints` |
| `C02`, `T03`, `T05` | политики, условия выполнены | `<yes>` | `search_policy_constraints` |
| `S01`, `S06` | сотрудник и команда | `<yes>`, `<no>` | `find_employee` |
| `S02`, `S05`, `S08` | руководитель или состав команды | `<yes>`, `<count:4>`, `<yes>` | `get_team` |
| `S03`, `S07`, `S09` | число задач по фильтрам | `<count:4>`, `<count:1>`, `<count:3>` | `search_tasks` |
| `S04`, `S10` | задача и исполнитель | `<yes>`, `<no>` | `search_tasks` |


In [ ]:
@dataclass(frozen=True)
class FactRule:
    description: str
    patterns: tuple[str, ...]


@dataclass(frozen=True)
class EvalCase:
    case_id: str
    split: Literal["regression", "single_source", "multistep"]
    question: str
    expected_placeholder: str
    required_fact_ids: tuple[str, ...]
    expected_functions: tuple[str, ...]
    slice: str


FACT_RULES: dict[str, FactRule] = {
    "vacation_limit": FactRule(
        "Перенести можно не более пяти дней.",
        (r"(?:не более|максимум).{0,25}(?:пят\w*|\b5\b)",),
    ),
    "written_manager_approval": FactRule(
        "Нужно письменное согласование руководителя.", (r"письмен", r"руководител")
    ),
    "sick_day_one": FactRule(
        "Сообщить нужно в первый день болезни.", (r"перв\w*\s+день",)
    ),
    "sick_recipients": FactRule(
        "Нужно уведомить руководителя и HR.", (r"руководител", r"(?:\bhr\b|эйчар)")
    ),
    "abroad_hr_approval": FactRule("Нужно согласование HR.", (r"(?:\bhr\b|эйчар)",)),
    "abroad_legal_approval": FactRule(
        "Нужно согласование юридической службы.", (r"юридическ",)
    ),
    "abroad_infosec_approval": FactRule(
        "Нужно согласование информационной безопасности.",
        (r"информационн.{0,20}безопасност",),
    ),
    "anna_agent_platform": FactRule(
        "Анна Петрова работает в Agent Platform.",
        (r"анн\w*\s+петров", r"agent platform"),
    ),
    "agent_platform_lead": FactRule(
        "Agent Platform руководит Илья Морозов.",
        (r"иль\w*\s+мороз", r"(?:руковод|лид)"),
    ),
    "support_eval_open_count": FactRule(
        "В Support Automation четыре открытые eval-задачи.",
        (r"(?:четыр\w*|\b4\b)", r"(?:eval|оценк)"),
    ),
    "maria_task_106": FactRule(
        "У Марии Соколовой в работе TASK-106.", (r"task-106", r"мар\w*\s+сокол")
    ),
    "agent_platform_size": FactRule(
        "В Agent Platform четыре сотрудника.",
        (r"(?:четыр\w*|\b4\b)", r"agent platform"),
    ),
    "artem_agent_platform": FactRule(
        "Артем Волков работает в Agent Platform.",
        (r"артем\w*\s+волк", r"agent platform"),
    ),
    "blocked_support_count": FactRule(
        "В Support Automation одна заблокированная задача.",
        (r"(?:одн\w*|\b1\b)", r"(?:заблок|blocked)"),
    ),
    "data_quality_lead": FactRule(
        "Data Quality руководит Ольга Лебедева.", (r"ольг\w*\s+лебед", r"data quality")
    ),
    "support_prompt_count": FactRule(
        "В Support Automation три prompt-задачи.", (r"(?:тр[ие]\w*|\b3\b)", r"prompt")
    ),
    "task_108_artem": FactRule(
        "TASK-108 назначена Артему Волкову.", (r"task-108", r"артем\w*\s+волк")
    ),
    "task_106_maria": FactRule(
        "TASK-106 назначена Марии Соколовой.", (r"task-106", r"мар\w*\s+сокол")
    ),
    "maria_agent_platform": FactRule(
        "Мария Соколова работает в Agent Platform.",
        (r"мар\w*\s+сокол", r"agent platform"),
    ),
    "agent_platform_prompt_open_count": FactRule(
        "У Agent Platform две открытые prompt-задачи.",
        (r"(?:дв[ае]\w*|\b2\b)", r"prompt"),
    ),
    "task_104_anna": FactRule(
        "TASK-104 назначена Анне Петровой.", (r"task-104", r"анн\w*\s+петров")
    ),
}

In [ ]:
REGRESSION_CASES = [
    EvalCase(
        "C01",
        "regression",
        "Можно ли перенести на следующий год десять неиспользованных дней отпуска?",
        "<no>",
        ("vacation_limit",),
        ("search_policy_constraints",),
        "превышение лимита",
    ),
    EvalCase(
        "C02",
        "regression",
        "Нужно ли в первый день болезни уведомить и непосредственного руководителя, и HR?",
        "<yes>",
        ("sick_day_one", "sick_recipients"),
        ("search_policy_constraints",),
        "срок и адресаты",
    ),
    EvalCase(
        "T01",
        "regression",
        "Можно ли перенести пять дней без предварительного письменного согласования непосредственного руководителя?",
        "<no>",
        ("written_manager_approval",),
        ("search_policy_constraints",),
        "обязательное согласование",
    ),
    EvalCase(
        "T02",
        "regression",
        "Достаточно ли согласования только HR, чтобы начать работать из другой страны?",
        "<no>",
        ("abroad_hr_approval", "abroad_legal_approval", "abroad_infosec_approval"),
        ("search_policy_constraints",),
        "несколько согласований",
    ),
    EvalCase(
        "T03",
        "regression",
        "Можно ли перенести на следующий год не более пяти дней, если заранее получено письменное согласование непосредственного руководителя?",
        "<yes>",
        ("vacation_limit", "written_manager_approval"),
        ("search_policy_constraints",),
        "все условия выполнены",
    ),
    EvalCase(
        "T04",
        "regression",
        "Достаточно ли при болезни написать только непосредственному руководителю на следующий день?",
        "<no>",
        ("sick_day_one", "sick_recipients"),
        ("search_policy_constraints",),
        "ложная предпосылка",
    ),
    EvalCase(
        "T05",
        "regression",
        "Нужно ли до работы из другой страны получить письменные согласования HR, юридической службы и службы информационной безопасности?",
        "<yes>",
        ("abroad_hr_approval", "abroad_legal_approval", "abroad_infosec_approval"),
        ("search_policy_constraints",),
        "полный список согласований",
    ),
    EvalCase(
        "T06",
        "regression",
        "Можно ли перенести шесть неиспользованных дней даже при письменном согласовании непосредственного руководителя?",
        "<no>",
        ("vacation_limit",),
        ("search_policy_constraints",),
        "лимит сильнее согласования",
    ),
]

SINGLE_SOURCE_CASES = [
    EvalCase(
        "S01",
        "single_source",
        "Работает ли Анна Петрова в команде Agent Platform?",
        "<yes>",
        ("anna_agent_platform",),
        ("find_employee",),
        "сотрудник",
    ),
    EvalCase(
        "S02",
        "single_source",
        "Верно ли, что Илья Морозов руководит командой Agent Platform?",
        "<yes>",
        ("agent_platform_lead",),
        ("get_team",),
        "руководитель команды",
    ),
    EvalCase(
        "S03",
        "single_source",
        "Сколько открытых задач проекта Support Automation имеют метку eval?",
        "<count:4>",
        ("support_eval_open_count",),
        ("search_tasks",),
        "точный count",
    ),
    EvalCase(
        "S04",
        "single_source",
        "Есть ли у Марии Соколовой задача в работе в проекте Support Automation?",
        "<yes>",
        ("maria_task_106",),
        ("search_tasks",),
        "исполнитель и статус",
    ),
    EvalCase(
        "S05",
        "single_source",
        "Сколько сотрудников входит в команду Agent Platform?",
        "<count:4>",
        ("agent_platform_size",),
        ("get_team",),
        "размер команды",
    ),
    EvalCase(
        "S06",
        "single_source",
        "Работает ли Артем Волков в команде Data Quality?",
        "<no>",
        ("artem_agent_platform",),
        ("find_employee",),
        "отрицательная проверка",
    ),
    EvalCase(
        "S07",
        "single_source",
        "Сколько заблокированных задач находится в проекте Support Automation?",
        "<count:1>",
        ("blocked_support_count",),
        ("search_tasks",),
        "точный статус",
    ),
    EvalCase(
        "S08",
        "single_source",
        "Руководит ли Ольга Лебедева командой Data Quality?",
        "<yes>",
        ("data_quality_lead",),
        ("get_team",),
        "руководитель команды",
    ),
    EvalCase(
        "S09",
        "single_source",
        "Сколько задач с меткой prompt есть в проекте Support Automation?",
        "<count:3>",
        ("support_prompt_count",),
        ("search_tasks",),
        "метка без статуса",
    ),
    EvalCase(
        "S10",
        "single_source",
        "Назначена ли задача TASK-108 Марии Соколовой?",
        "<no>",
        ("task_108_artem",),
        ("search_tasks",),
        "задача по id",
    ),
]

EVAL_18 = REGRESSION_CASES + SINGLE_SOURCE_CASES
assert len(EVAL_18) == 18

In [ ]:
case_ids_18 = [case.case_id for case in EVAL_18]
assert len(case_ids_18) == len(set(case_ids_18)) == 18
assert all(case.required_fact_ids for case in EVAL_18)
assert all(case.expected_functions for case in EVAL_18)

### Зачем нужен `<count:n>`

Для пользователя такой маркер выглядит немного искусственно. Для эксперимента он удобен: основной результат можно проверить отдельно от объяснения.

В вопросе на количество модель способна перечислить правильные фильтры и всё равно написать неверное число. `<count:4>` ловит эту ошибку простым парсером.

In [ ]:
PLACEHOLDER_RE = re.compile(
    r"^\s*(<yes>|<no>|<count:\d+>)\s+(.+?)\s*$", flags=re.DOTALL | re.IGNORECASE
)


def parse_answer_contract(answer: Optional[str]) -> dict[str, Any]:
    if not answer:
        return {
            "placeholder": None,
            "body": None,
            "syntax_valid": False,
            "error": "Пустой итоговый ответ.",
        }

    match = PLACEHOLDER_RE.fullmatch(answer)
    if match is None:
        return {
            "placeholder": None,
            "body": None,
            "syntax_valid": False,
            "error": "Ответ должен начинаться с <yes>, <no> или <count:n> и содержать объяснение.",
        }

    return {
        "placeholder": match.group(1).lower(),
        "body": match.group(2).strip(),
        "syntax_valid": True,
        "error": None,
    }


contract_examples = [
    "<yes> Анна Петрова работает в Agent Platform.",
    "<count:4> Найдены четыре открытые задачи с меткой eval.",
    "Четыре задачи.",
    "<no>",
]
show_table(
    pd.DataFrame(
        [
            {"answer": answer, **parse_answer_contract(answer)}
            for answer in contract_examples
        ]
    )
)

## Три способа собрать систему вокруг одних функций

Данные и функции можно оставить прежними, а управление шагами устроить по-разному.

| Подход | Кто выбирает шаги | Что удобно | Где появляется цена |
|---|---|---|---|
| workflow | приложение | порядок виден в коде, легко отлаживать | каждый маршрут нужно описать заранее |
| SGR next-step | модель возвращает структурированный следующий шаг | действие и аргументы проходят строгую проверку | приложение поддерживает собственный протокол, модель должна быть хорошая в SO |
| ReAct через функции | модель выбирает функцию через API | цикл естественно строится поверх function calling, современные модели обучаются вокруг этого паттерна | история вызовов быстро растёт |

Дальше соберём SGR и ReAct на одинаковых функциях и кейсах.


diagram

*Схема: SGR next-step, нативный function calling и ReAct-цикл над одними данными и функциями*

## Структурированный агент SGR next-step

Schema-Guided Reasoning задаёт промежуточный результат модели схемой. Под этим названием скрывается семейство вариантов. В [демо SGR](https://abdullin.com/schema-guided-reasoning/demo) используется планировщик `NextStep`: он выбирает одно действие и снова решает, что делать, после каждого нового результата.

Наша версия работает так:

```text
вопрос + история
        |
        v
NextStep: выбрать одно действие
        |
        v
схема аргументов выбранного действия
        |
        +--> final_answer: закончить цикл
        |
        +--> функция: исполнить и добавить результат в историю
```

Один логический шаг занимает до двух модельных вызовов. Сначала выбирается действие, затем отдельная схема собирает его аргументы. Приложение валидирует оба объекта, вызывает Python и записывает результат в траекторию.

Цена подхода уже видна: схемы отдельных действий упрощают проверку, но добавляют запросы и токены. Позже сравним это с нативным вызовом функций.


In [ ]:
class SearchPolicyAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: Literal["search_policy_constraints"]
    query: str
    limit: int = Field(default=3, ge=1, le=5)


class FindEmployeeAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: Literal["find_employee"]
    query: str


class GetTeamAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: Literal["get_team"]
    team: str


class SearchTasksAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: Literal["search_tasks"]
    query: Optional[str] = None
    project: Optional[str] = None
    status: Optional[Literal["open", "in_progress", "blocked", "done"]] = None
    is_open: Optional[bool] = None
    assignee: Optional[str] = None
    labels: Optional[list[str]] = None
    limit: int = Field(default=20, ge=1, le=50)


class FinalAnswerAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: Literal["final_answer"]
    answer: str = Field(min_length=1)


AgentAction = Union[
    SearchPolicyAction,
    FindEmployeeAction,
    GetTeamAction,
    SearchTasksAction,
    FinalAnswerAction,
]


class NextStep(BaseModel):
    model_config = ConfigDict(extra="forbid")
    action_name: Literal[
        "search_policy_constraints",
        "find_employee",
        "get_team",
        "search_task_by_id_or_title",
        "filter_tasks_by_project",
        "filter_tasks_by_assignee",
        "final_answer",
    ]


class SearchPolicyArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    query: str
    limit: int = Field(ge=1, le=5)


class FindEmployeeArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    query: str


class GetTeamArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    team: str


class SearchTaskByIdArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    task_id_or_title: str


class FilterTasksByProjectArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    project: Optional[str]
    state_filter: Optional[
        Literal["open", "in_progress", "blocked", "done", "not_done"]
    ]
    label: Optional[str]


class FilterTasksByAssigneeArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    assignee: str
    project: Optional[str]
    state_filter: Optional[
        Literal["open", "in_progress", "blocked", "done", "not_done"]
    ]


class FinalAnswerArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    answer: str = Field(min_length=1)


def argument_model_for_action(action_name: str) -> Type[BaseModel]:
    if action_name == "search_policy_constraints":
        return SearchPolicyArguments
    if action_name == "find_employee":
        return FindEmployeeArguments
    if action_name == "get_team":
        return GetTeamArguments
    if action_name == "search_task_by_id_or_title":
        return SearchTaskByIdArguments
    if action_name == "filter_tasks_by_project":
        return FilterTasksByProjectArguments
    if action_name == "filter_tasks_by_assignee":
        return FilterTasksByAssigneeArguments
    if action_name == "final_answer":
        return FinalAnswerArguments
    raise ValueError(f"Неизвестное действие: {action_name}")


def build_action_from_arguments(
    action_name: str,
    arguments: BaseModel,
) -> AgentAction:
    data = arguments.model_dump(mode="json", exclude_none=True)

    if action_name == "search_policy_constraints":
        return SearchPolicyAction.model_validate({"name": action_name, **data})
    if action_name == "find_employee":
        return FindEmployeeAction.model_validate({"name": action_name, **data})
    if action_name == "get_team":
        return GetTeamAction.model_validate({"name": action_name, **data})
    if action_name == "final_answer":
        return FinalAnswerAction.model_validate({"name": action_name, **data})

    task_query = data.pop("task_id_or_title", None)
    state_filter = data.pop("state_filter", None)
    label = data.pop("label", None)
    payload: dict[str, Any] = {
        "name": "search_tasks",
        "query": task_query,
        **data,
    }
    if state_filter == "not_done":
        payload["is_open"] = True
    elif state_filter is not None:
        payload["status"] = state_filter
    if label is not None:
        payload["labels"] = [label]
    return SearchTasksAction.model_validate(payload)


choice_example = NextStep(action_name="filter_tasks_by_project")
arguments_example = FilterTasksByProjectArguments(
    project="Support Automation",
    state_filter="not_done",
    label="eval",
)
action_example = build_action_from_arguments(
    choice_example.action_name, arguments_example
)
assert isinstance(action_example, SearchTasksAction)
assert action_example.is_open is True
assert action_example.labels == ["eval"]

argument_schema_sizes = {
    model.__name__: len(json.dumps(model.model_json_schema(), ensure_ascii=False))
    for model in (
        SearchTaskByIdArguments,
        FilterTasksByProjectArguments,
        FilterTasksByAssigneeArguments,
    )
}
print(
    {
        "choice_schema_chars": len(
            json.dumps(NextStep.model_json_schema(), ensure_ascii=False)
        ),
        "largest_arguments_schema_chars": max(argument_schema_sizes.values()),
    }
)

### Зачем мы делим выбор и аргументы

Можно было передать модели одну большую `Union`-схему со всеми действиями и всеми полями. В учебной реализации выбран другой вариант: после `NextStep` модель видит только аргументы конкретного действия.

Для поиска задач получаются три небольшие схемы:

- поиск по ID или словам из заголовка;
- фильтрация по проекту, состоянию и одной метке;
- фильтрация по исполнителю, проекту и состоянию.

Приложение затем приводит их к общей `SearchTasksAction`. Значение `not_done` превращается в `is_open=True`, одна метка становится списком из одного элемента.

Такая каскадная схема не гарантирует лучший результат сама по себе. Она локализует ошибку локальнее: отдельно видно неверный выбор действия и отдельно неверные аргументы. За эту наблюдаемость мы платим дополнительным LLM-вызовом.


In [ ]:
SGR_SYSTEM_PROMPT = """
Ты корпоративный помощник. На каждом шаге выбери ровно одно следующее действие.

Действия:
- search_policy_constraints: проверить правило или ограничение;
- find_employee: найти одного сотрудника и его текущую команду;
- get_team: получить руководителя и участников команды;
- search_task_by_id_or_title: найти задачу по ID или словам из заголовка;
- filter_tasks_by_project: отфильтровать задачи по проекту, состоянию и одной метке;
- filter_tasks_by_assignee: отфильтровать задачи одного исполнителя по проекту и состоянию;
- final_answer: ответить, когда данных уже достаточно.

Не выдумывай данные. Для количества используй count или member_count из результата действия.
Сейчас верни только action_name. Аргументы приложение запросит отдельной схемой выбранного действия.
""".strip()


def argument_instruction_for_action(action_name: str) -> str:
    if action_name == "search_policy_constraints":
        return "Заполни query для поиска правила и limit от 1 до 5. Обычно достаточно limit=3."
    if action_name == "find_employee":
        return "Заполни query именем, рабочей почтой или employee_id одного сотрудника."
    if action_name == "get_team":
        return "Заполни team точным названием или team_id команды."
    if action_name == "search_task_by_id_or_title":
        return "Заполни task_id_or_title точным ID или словами из заголовка задачи."
    if action_name == "filter_tasks_by_project":
        return (
            "Заполни project, state_filter и label. Для всех незавершённых задач выбери "
            "state_filter=not_done, для точного статуса выбери open, in_progress, blocked или done. "
            "Ненужные поля верни null."
        )
    if action_name == "filter_tasks_by_assignee":
        return (
            "Заполни assignee одним сотрудником, project и state_filter. Для всех незавершённых задач "
            "выбери state_filter=not_done, для точного статуса выбери open, in_progress, blocked или done. "
            "Ненужные поля верни null."
        )
    if action_name == "final_answer":
        return (
            "Верни итоговый answer. Он начинается ровно с <yes>, <no> или <count:n>, "
            "затем идёт короткое проверяемое объяснение по полученным данным."
        )
    raise ValueError(f"Неизвестное действие: {action_name}")


def text_message(role: str, text: str) -> dict[str, Any]:
    return {"role": role, "content": [{"text": text}]}


def usage_from_response(response: Any) -> dict[str, int]:
    """Возвращает обработанные токены, включая часть запроса из кэша."""
    usage = getattr(response, "usage", None)
    if usage is None:
        raise RuntimeError("GigaChat не вернул сведения об использовании токенов.")
    data = (
        usage.model_dump(mode="json", exclude_none=True)
        if hasattr(usage, "model_dump")
        else dict(usage)
    )
    if "prompt_tokens" in data:
        prompt = int(data["prompt_tokens"] or 0) + int(
            data.get("precached_prompt_tokens", 0) or 0
        )
    elif "input_tokens" in data:
        input_details = data.get("input_tokens_details") or {}
        prompt = int(data["input_tokens"] or 0) + int(
            input_details.get("cached_tokens", 0) or 0
        )
    else:
        raise RuntimeError("GigaChat не вернул счётчик входных токенов.")
    completion = int(data.get("completion_tokens", data.get("output_tokens", 0)) or 0)
    if prompt < 0 or completion < 0:
        raise RuntimeError(f"GigaChat вернул некорректные счётчики токенов: {data}")
    total = prompt + completion
    return {
        "prompt_tokens": prompt,
        "completion_tokens": completion,
        "total_tokens": total,
    }


def dispatch_tool(
    name: str,
    arguments: dict[str, Any],
    handlers: Optional[dict[str, Callable[..., dict[str, Any]]]] = None,
) -> dict[str, Any]:
    selected_handlers = handlers or TOOL_HANDLERS
    if name not in selected_handlers:
        raise KeyError(f"Неизвестная функция: {name}")
    return selected_handlers[name](**arguments)

In [ ]:
@dataclass
class AgentRun:
    architecture: str
    final_answer: str
    called_functions: list[str]
    steps: int
    prompt_tokens: int
    completion_tokens: int
    total_tokens: int
    latency_seconds: float
    trace: list[dict[str, Any]] = field(default_factory=list)
    error: Optional[str] = None


def run_sgr_agent(
    question: str,
    *,
    max_steps: int = 8,
    step_model: Type[BaseModel] = NextStep,
    system_prompt: str = SGR_SYSTEM_PROMPT,
    argument_model_resolver: Callable[
        [str], Type[BaseModel]
    ] = argument_model_for_action,
    argument_instruction_resolver: Callable[
        [str], str
    ] = argument_instruction_for_action,
    action_builder: Callable[[str, BaseModel], BaseModel] = build_action_from_arguments,
    initial_action_name: Optional[str] = None,
    handlers: Optional[dict[str, Callable[..., dict[str, Any]]]] = None,
) -> AgentRun:
    history = [
        text_message("system", system_prompt),
        text_message("user", question),
    ]
    trace: list[dict[str, Any]] = []
    called_functions: list[str] = []
    token_totals = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    started = time.perf_counter()

    for step_number in range(1, max_steps + 1):
        if step_number == 1 and initial_action_name is not None:
            choice_completion = None
            parsed_step = step_model.model_validate(
                {"action_name": initial_action_name}
            )
        else:
            choice_request = {
                "messages": history,
                "model_options": {"temperature": 0.0, "max_tokens": 80},
            }
            choice_completion, parsed_step = client.chat.parse(
                choice_request,
                response_format=step_model,
                strict=True,
            )
        action_name = parsed_step.action_name

        arguments_request = {
            "messages": history
            + [
                text_message("assistant", parsed_step.model_dump_json()),
                text_message("user", argument_instruction_resolver(action_name)),
            ],
            "model_options": {"temperature": 0.0, "max_tokens": 300},
        }
        arguments_completion, parsed_arguments = client.chat.parse(
            arguments_request,
            response_format=argument_model_resolver(action_name),
            strict=True,
        )

        completions = [arguments_completion]
        if choice_completion is not None:
            completions.insert(0, choice_completion)
        for completion in completions:
            usage = usage_from_response(completion)
            for key in token_totals:
                token_totals[key] += usage[key]

        arguments_payload = parsed_arguments.model_dump(mode="json", exclude_none=True)
        step_payload = {
            "action_name": action_name,
            "arguments": arguments_payload,
        }
        trace.append(
            {"kind": "model_step", "step": step_number, "payload": step_payload}
        )
        history.append(
            text_message("assistant", json.dumps(step_payload, ensure_ascii=False))
        )

        action = action_builder(action_name, parsed_arguments)
        if isinstance(action, FinalAnswerAction):
            return AgentRun(
                architecture="sgr_next_step",
                final_answer=action.answer,
                called_functions=called_functions,
                steps=step_number,
                latency_seconds=time.perf_counter() - started,
                trace=trace,
                **token_totals,
            )

        action_data = action.model_dump(mode="json", exclude_none=True)
        function_name = action_data.pop("name")
        result = dispatch_tool(function_name, action_data, handlers)
        called_functions.append(function_name)
        trace.append(
            {
                "kind": "tool_result",
                "step": step_number,
                "name": function_name,
                "arguments": action_data,
                "result": result,
            }
        )
        history.append(
            text_message(
                "user",
                f'<function_result name="{function_name}">\n'
                + json.dumps(result, ensure_ascii=False)
                + "\n</function_result>",
            )
        )

    raise RuntimeError(f"SGR превысил лимит {max_steps} шагов.")

### Первый живой вопрос

Начнём с простого кейса `S03`: нужно посчитать открытые задачи проекта `Support Automation` с меткой `eval`.

В траектории ожидается один вызов `search_tasks`, затем `final_answer`. Это пока проверка механики. Полный оценочный прогон появится после того, как обе архитектуры будут собраны.


In [ ]:
sgr_demo = run_sgr_agent(
    "Сколько открытых задач проекта Support Automation имеют метку eval?"
)
print(sgr_demo.final_answer)
show_table(pd.DataFrame(sgr_demo.trace))

## От своей схемы к нативному вызову функций

SGR уже работает, но протокол между моделью и приложением мы придумали сами. Модель сначала заполняет `NextStep`, затем получает вторую схему аргументов, а функция вызывается нашим кодом.

В API моделей обычно есть готовый канал для той же задачи. Модель может вернуть `function_call` с именем и аргументами. Python по-прежнему исполняет функцию, модель получает только результат.

Полный проход вызова функции выглядит так:

```text
вопрос + список функций
          |
          v
модель возвращает function_call
          |
          v
приложение проверяет имя и аргументы
          |
          v
Python вызывает функцию
          |
          v
function_result возвращается модели
          |
          v
модель пишет итоговый ответ
```

Для `S03` запрос функции может выглядеть так:

```json
{
  "name": "search_tasks",
  "arguments": {
    "project": "Support Automation",
    "is_open": true,
    "labels": ["eval"]
  }
}
```

Важно не приписывать этому объекту лишнюю магию. На момент получения `function_call` никакой Python ещё не запущен. Доступ к данным, проверка разрешений и обработка ошибок остаются в приложении.


Небольшая ремарка, современные harness-слов как раз про обработку вызова функций и обработку ее результата. Предоставления доступа к данным, сокращения контекста, подсказки, бюджеты – там море подходов.

### Нативный function calling в GigaChat

Ноутбук закрепляет `gigachat==0.2.3` и передаёт настройки через `tool_config`.

| `mode` | Поведение | Дополнительное поле |
|---|---|---|
| `none` | модель отвечает текстом и не выбирает функции | нет |
| `auto` | модель сама выбирает текст или одну из переданных функций | нет |
| `forced` | модель обязана запросить указанную функцию | `function_name` |

Для обычного вопроса подходит `auto`. Режим `forced` понадобится позже, когда эксперимент с планировщиком потребует начать запуск с `update_plan`.

В актуальной REST-документации GigaChat похожая политика описывается через поле `function_call`. Между версиями SDK поля могут отличаться, поэтому здесь ориентируемся на закреплённую версию и её модели запросов.

Ответ с вызовом содержит `function_call` и идентификатор состояния `tools_state_id`. В следующий запрос переносится сообщение ассистента с вызовом и сообщение роли `tool` с `function_result`. Эти сообщения образуют пару. Удаление одной половины ломает историю протокола.

Такая последовательность показана в [примере function calling GigaChat SDK](https://github.com/ai-forever/gigachat/blob/main/examples/tools/function_calling.py). Перед исполнением имя проверяется по `TOOL_HANDLERS`, поэтому модель не может вызвать произвольную Python-функцию по строке.


In [ ]:
def response_to_dict(response: Any) -> dict[str, Any]:
    if hasattr(response, "model_dump"):
        return response.model_dump(mode="json", exclude_none=True, by_alias=True)
    return dict(response)


def first_message_text(response: Any) -> str:
    payload = response_to_dict(response)
    for message in payload.get("messages", []):
        content = message.get("content") or []
        if isinstance(content, str):
            return content
        for part in content:
            if isinstance(part, dict) and part.get("text"):
                return str(part["text"])
    return ""


def extract_function_call(response: Any) -> Optional[dict[str, Any]]:
    payload = response_to_dict(response)
    for message in payload.get("messages", []):
        function_call = message.get("function_call")
        if function_call is None:
            for part in message.get("content") or []:
                if isinstance(part, dict) and part.get("function_call") is not None:
                    function_call = part["function_call"]
                    break
        if function_call is None:
            continue

        arguments = function_call.get("arguments") or {}
        if isinstance(arguments, str):
            arguments = json.loads(arguments)

        state_field = next(
            (
                field_name
                for field_name in (
                    "tools_state_id",
                    "tool_state_id",
                    "functions_state_id",
                )
                if message.get(field_name) is not None
            ),
            None,
        )
        state_id = message.get(state_field) if state_field else None
        assistant_message = copy.deepcopy(message)
        assistant_message.pop("tool_state_id", None)
        assistant_message.pop("functions_state_id", None)
        if state_id is not None:
            assistant_message["tools_state_id"] = state_id

        return {
            "name": function_call["name"],
            "arguments": arguments,
            "state_field": "tools_state_id" if state_id is not None else None,
            "state_id": state_id,
            "assistant_message": assistant_message,
        }
    return None


def append_function_result(
    request: dict[str, Any],
    call: dict[str, Any],
    result: dict[str, Any],
) -> dict[str, Any]:
    tool_message: dict[str, Any] = {
        "role": "tool",
        "content": [
            {
                "function_result": {
                    "name": call["name"],
                    "result": result,
                }
            }
        ],
    }
    if call["state_field"] is not None:
        tool_message[call["state_field"]] = call["state_id"]

    updated = copy.deepcopy(request)
    updated.pop("tool_config", None)
    updated["messages"] = request["messages"] + [
        call["assistant_message"],
        tool_message,
    ]
    return updated


TOOL_CALL_BASELINE_SYSTEM_PROMPT = """
Ты корпоративный помощник с доступом к функциям.
Получай актуальные данные через функции и не придумывай значения.
Для количества используй count из результата функции.
Итоговый ответ начинается с <yes>, <no> или <count:n>, затем идёт короткое объяснение.
""".strip()

function_demo_request = {
    "messages": [
        text_message("system", TOOL_CALL_BASELINE_SYSTEM_PROMPT),
        text_message(
            "user",
            "Сколько открытых задач проекта Support Automation имеют метку eval?",
        ),
    ],
    "tools": [{"functions": {"specifications": TOOL_SPECS}}],
    "tool_config": {"mode": "auto"},
    "model_options": {"temperature": 0.01, "max_tokens": 500},
}
assert "reasoning" not in function_demo_request["model_options"]

function_demo_response = client.chat.create(function_demo_request)
function_demo_call = extract_function_call(function_demo_response)
if function_demo_call is None:
    raise RuntimeError("GigaChat ответил текстом вместо вызова функции.")
if function_demo_call["name"] != "search_tasks":
    raise RuntimeError(f"Ожидался search_tasks, получено {function_demo_call['name']}.")

function_demo_result = dispatch_tool(
    function_demo_call["name"],
    function_demo_call["arguments"],
)
function_demo_final_response = client.chat.create(
    append_function_result(
        function_demo_request, function_demo_call, function_demo_result
    )
)
function_demo_answer = first_message_text(function_demo_final_response)
if not function_demo_answer:
    raise RuntimeError("После результата функции GigaChat не вернул итоговый текст.")

print("Модель запросила:")
print(
    json.dumps(
        {
            "name": function_demo_call["name"],
            "arguments": function_demo_call["arguments"],
        },
        ensure_ascii=False,
        indent=2,
    )
)
print("Результат приложения: count =", function_demo_result["count"])
print("Итог модели:", function_demo_answer)

### Разбираем один проход

Вывод ячейки показывает три разных события:

- модель запросила `search_tasks` и сформировала аргументы;
- приложение исполнило функцию и получило `count`;
- второй вызов модели превратил результат функции в итоговый ответ.

Системная инструкция здесь короткая. В ней нет просьбы описывать рассуждение или заранее расписывать маршрут. Модель просто выбирает доступную функцию.

Пока приложение выполняет один проход. Оно ожидает один вызов функции и после результата сразу просит ответ. Дальше будем повторять эту механику, пока модель не вернёт текст вместо `function_call`.


## Повторяем вызовы в ReAct-цикле

В `S03` одного поиска достаточно. С вопросом про `TASK-106` так не получится. Сначала надо найти связанную сущность, затем перейти к сотруднику и команде, а следующий вызов зависит от результата предыдущего.

Если после каждого `function_result` снова обращаться к модели, один проход превращается в цикл:

```text
вопрос -> function_call -> function_result -> новый вызов модели
             ^                                  |
             +----------------------------------+
```

От работы [ReAct](https://arxiv.org/abs/2210.03629) здесь берём чередование действия и наблюдения. В исходной статье модель также печатала промежуточные `Thought`. В ноутбуке они не запрашиваются и не сохраняются. Для приложения достаточно функции, аргументов, результата и финального ответа.

В статье ReAct авторы сообщали прирост абсолютной доли успеха на 34 и 10 процентных пунктов на ALFWorld и WebShop относительно сравниваемых методов. Наш трекер, модель и задачи устроены иначе. Поэтому дальше смотрим на собственную траекторию, а не переносим цифры из бенчмарка на учебного агента.


In [ ]:
REACT_SYSTEM_PROMPT = """
Ты корпоративный помощник с доступом к функциям.

Правила:
1. Получай актуальные данные через функции. Не придумывай сотрудников, команды, задачи и политики.
2. Для количества используй count или member_count из результата функции.
3. Если вопрос связывает сотрудника, команду и задачи, вызывай функции последовательно.
4. Не вызывай функцию повторно с теми же аргументами без причины.
5. Когда данных достаточно, верни итоговый ответ. Он начинается ровно с <yes>, <no> или <count:n>, затем идёт короткое проверяемое объяснение.
""".strip()


def build_react_request(
    question: str,
    *,
    tool_specs: Optional[list[dict[str, Any]]] = None,
    system_prompt: str = REACT_SYSTEM_PROMPT,
) -> dict[str, Any]:
    return {
        "messages": [
            text_message("system", system_prompt),
            text_message("user", question),
        ],
        "tools": [{"functions": {"specifications": tool_specs or TOOL_SPECS}}],
        "tool_config": {"mode": "auto"},
        "model_options": {
            "temperature": 0.01,
            "max_tokens": 500,
        },
    }


example_request = build_react_request("Сколько открытых задач имеют метку eval?")
assert "reasoning" not in example_request["model_options"]
print(json.dumps(example_request, ensure_ascii=False, indent=2)[:2600])

### Вопрос, где наблюдение меняет следующий шаг

Проверим утверждение про `TASK-106`, её исполнителя и руководителя команды.

Для ответа понадобятся `search_tasks`, `find_employee` и `get_team`. Порядок заранее не фиксируем: модель может начать с задачи или с руководителя. Важно, чтобы в конце появились нужные факты и проверяемая связь между ними.

После прогона смотрим на три поля:

- `called_functions` показывает фактический маршрут;
- `steps` показывает число обращений к модели;
- `total_tokens` суммирует стоимость всех шагов.


### Оборачиваем function calling в цикл

`run_react_agent` повторяет одну и ту же механику до финального текста:

- отправляет текущий запрос модели;
- извлекает `function_call`;
- проверяет и исполняет функцию;
- добавляет вызов и результат в историю;
- останавливается по текстовому ответу или по лимиту шагов.

В учебной версии жёстким предохранителем служит `max_steps`. Правило грубое, зато его легко увидеть и проверить. В рабочей системе остановка зависит от задачи: можно ловить повтор одинаковых вызовов, отсутствие прогресса, исчерпание бюджета или проверять готовность отдельным оценщиком. Лимит шагов всё равно остаётся последней защитой.

Каждый результат записывается в отдельную `trace`. Сжатие истории в демонстрационном прогоне отключено, чтобы сначала увидеть цикл целиком.


In [ ]:
def run_react_agent(
    question: str,
    *,
    max_steps: int = 8,
    compact_after_proxy_units: Optional[int] = 6000,
    tool_specs: Optional[list[dict[str, Any]]] = None,
    system_prompt: str = REACT_SYSTEM_PROMPT,
    first_function_name: Optional[str] = None,
    handlers: Optional[dict[str, Callable[..., dict[str, Any]]]] = None,
) -> AgentRun:
    request = build_react_request(
        question,
        tool_specs=tool_specs,
        system_prompt=system_prompt,
    )
    trace: list[dict[str, Any]] = []
    called_functions: list[str] = []
    token_totals = {"prompt_tokens": 0, "completion_tokens": 0, "total_tokens": 0}
    started = time.perf_counter()

    for step_number in range(1, max_steps + 1):
        if step_number == 1 and first_function_name is not None:
            request["tool_config"] = {
                "mode": "forced",
                "function_name": first_function_name,
            }
        response = client.chat.create(request)
        usage = usage_from_response(response)
        for key in token_totals:
            token_totals[key] += usage[key]

        call = extract_function_call(response)
        if step_number == 1 and first_function_name is not None:
            if call is None or call["name"] != first_function_name:
                raise RuntimeError(
                    f"GigaChat не выполнил обязательный первый вызов {first_function_name}."
                )
        if call is None:
            answer = first_message_text(response)
            if not answer:
                raise RuntimeError(
                    "GigaChat не вернул ни вызов функции, ни итоговый ответ."
                )
            trace.append(
                {"kind": "final_answer", "step": step_number, "answer": answer}
            )
            return AgentRun(
                architecture="react_function_calling",
                final_answer=answer,
                called_functions=called_functions,
                steps=step_number,
                latency_seconds=time.perf_counter() - started,
                trace=trace,
                **token_totals,
            )

        result = dispatch_tool(call["name"], call["arguments"], handlers)
        called_functions.append(call["name"])
        trace.append(
            {
                "kind": "tool_result",
                "step": step_number,
                "name": call["name"],
                "arguments": call["arguments"],
                "result": result,
            }
        )
        request = append_function_result(request, call, result)
        if step_number == 1 and first_function_name is not None:
            request["tool_config"] = {"mode": "auto"}

        if (
            compact_after_proxy_units is not None
            and proxy_units(request["messages"]) > compact_after_proxy_units
        ):
            request["messages"] = compact_native_messages(request["messages"], trace)
            trace.append(
                {
                    "kind": "context_compaction",
                    "step": step_number,
                    "working_proxy_units": proxy_units(request["messages"]),
                }
            )

    raise RuntimeError(f"ReAct превысил лимит {max_steps} шагов.")

In [ ]:
react_demo = run_react_agent(
    "Верно ли, что TASK-106 назначена сотруднику команды, которой руководит Илья Морозов?",
    compact_after_proxy_units=None,
)
print(react_demo.final_answer)
print(
    {
        "called_functions": react_demo.called_functions,
        "steps": react_demo.steps,
        "total_tokens": react_demo.total_tokens,
    }
)
show_table(pd.DataFrame(react_demo.trace))

assert {"search_tasks", "find_employee", "get_team"}.issubset(
    react_demo.called_functions
)

### Что дал цикл и за что пришлось заплатить

Теперь приложение не хранит маршрут `задача -> сотрудник -> команда` в коде. Модель собирает его по наблюдениям. Это полезная часть агентности: следующий шаг определяется текущим состоянием, а не заранее выбранной веткой `if`.

Вместе с гибкостью появились расходы. Один пользовательский вопрос породил несколько модельных вызовов. Модель может начать с менее удобной сущности, проверить лишнюю гипотезу или повторить поиск с другими аргументами.

Таблица траектории делает это видимым. Если одна функция встречается дважды, проблема уже конкретная: можно разбирать описание инструмента, аргументы первого вызова или состав результата. По финальной фразе такую причину обычно не восстановить.

Есть ещё один эффект. На каждом шаге модели снова отправляется старая история. После нескольких крупных результатов трекера она начинает занимать больше места, чем сам вопрос.


### Почему история быстро раздувается

Запрос на шаге `k` содержит инструкции, схемы функций, вопрос пользователя и все предыдущие пары `function_call` плюс `function_result`.

```text
request_k = instructions + tools + question + previous_tool_pairs
```

Если `search_tasks` вернул двадцать задач на втором шаге, эти двадцать задач снова поедут в третий, четвёртый и пятый вызовы. На короткой синтетике это терпимо. В рабочем трекере получается накопительный налог на каждое наблюдение.

Полная траектория всё равно нужна для аудита и разбора ошибок. На следующий шаг модели нужен отобранный срез: важные факты, последние пары и ссылки, по которым тяжёлый объект можно запросить снова.

Следующая ячейка собирает простую версию такого сжатия.


In [ ]:
TOKEN_PROXY_RE = re.compile(r"\w+|[^\w\s]", flags=re.UNICODE)


def proxy_units(value: Any) -> int:
    """Локальная единица сравнения. Это не токены GigaChat."""
    text = value if isinstance(value, str) else json.dumps(value, ensure_ascii=False)
    return len(TOKEN_PROXY_RE.findall(text))


def trace_summary(events: list[dict[str, Any]]) -> str:
    lines = []
    for event in events:
        if event.get("kind") != "tool_result":
            continue
        compact_result = json.dumps(event["result"], ensure_ascii=False)
        if len(compact_result) > 700:
            compact_result = compact_result[:700] + "..."
        lines.append(
            f"{event['name']}({json.dumps(event['arguments'], ensure_ascii=False)}) -> {compact_result}"
        )
    return "\n".join(lines)


def compact_native_messages(
    messages: list[dict[str, Any]],
    trace: list[dict[str, Any]],
    *,
    keep_last_tool_pairs: int = 2,
) -> list[dict[str, Any]]:
    tool_events = [item for item in trace if item.get("kind") == "tool_result"]
    if len(tool_events) <= keep_last_tool_pairs:
        return messages

    old_events = tool_events[:-keep_last_tool_pairs]
    pair_messages_to_keep = 2 * keep_last_tool_pairs
    base_messages = messages[:2]
    recent_messages = messages[-pair_messages_to_keep:]
    summary_message = text_message(
        "user",
        "<execution_summary>\n" + trace_summary(old_events) + "\n</execution_summary>",
    )
    return base_messages + [summary_message] + recent_messages

In [ ]:
# Готовим искусственно длинную историю, чтобы проверить механику без API.
synthetic_messages = [
    text_message("system", REACT_SYSTEM_PROMPT),
    text_message("user", "Учебный длинный вопрос"),
]
synthetic_trace = []
for index in range(6):
    synthetic_messages.extend(
        [
            {
                "role": "assistant",
                "content": [
                    {
                        "function_call": {
                            "name": "search_tasks",
                            "arguments": {"query": f"TASK-{index}"},
                        }
                    }
                ],
            },
            {
                "role": "tool",
                "content": [
                    {
                        "function_result": {
                            "name": "search_tasks",
                            "result": {"items": [{"title": "x" * 180}], "count": 1},
                        }
                    }
                ],
            },
        ]
    )
    synthetic_trace.append(
        {
            "kind": "tool_result",
            "name": "search_tasks",
            "arguments": {"query": f"TASK-{index}"},
            "result": {"items": [{"title": "x" * 180}], "count": 1},
        }
    )

compacted_messages = compact_native_messages(
    synthetic_messages, synthetic_trace, keep_last_tool_pairs=2
)
comparison = pd.DataFrame(
    [
        {
            "version": "full",
            "messages": len(synthetic_messages),
            "proxy_units": proxy_units(synthetic_messages),
        },
        {
            "version": "compacted",
            "messages": len(compacted_messages),
            "proxy_units": proxy_units(compacted_messages),
        },
    ]
)
show_table(comparison)

assert len(compacted_messages) == 2 + 1 + 4
assert compacted_messages[-1]["role"] == "tool"

### Сжатие без разрыва протокола

В демонстрации используется локальная величина `proxy_units`. Это счётчик слов и знаков, а не токены GigaChat. Он нужен только для сравнения полной и сокращённой истории без дополнительного API-вызова.

Сообщение ассистента с `function_call` не остаётся без соответствующего `tool`-результата. Факты из старых результатов собираются из структурированной траектории в короткую сводку, последние пары сохраняются буквально.

Подход близок к `just-in-time` контексту из статьи Anthropic [Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents): в активном окне остаются лёгкие ссылки и нужные факты, а тяжёлые данные загружаются инструментами по мере необходимости. В нашем наборе такими ссылками служат `task_id`, `employee_id` и `team_id`.

Эта учебная функция не доказывает, что сводка сохраняет всё важное на длинных задачах. Она показывает место, где стратегию можно тестировать отдельно от агента.

## Собираем оценочный контур агента

После появления цикла одного финального ответа мало. Для каждого прогона сохраняем:

- ожидаемый и фактический плейсхолдер;
- долю найденных обязательных фактов;
- ожидаемые и реально вызванные функции;
- число модельных шагов и вызовов инструментов;
- токены и задержку;
- финальный текст, ошибку и полную траекторию.

Метрики лежат в отдельных колонках. Правильный ответ после десяти повторных поисков не смешивается с короткой и аккуратной траекторией.


In [ ]:
def fact_is_present(answer: str, fact_id: str) -> bool:
    normalized = answer.lower().replace("ё", "е")
    rule = FACT_RULES[fact_id]
    return all(
        re.search(pattern, normalized, flags=re.IGNORECASE | re.DOTALL)
        for pattern in rule.patterns
    )


def evaluate_agent_run(case: EvalCase, run: AgentRun) -> dict[str, Any]:
    parsed = parse_answer_contract(run.final_answer)
    fact_checks = {
        fact_id: fact_is_present(run.final_answer, fact_id)
        for fact_id in case.required_fact_ids
    }
    expected_functions = set(case.expected_functions)
    called_functions = set(run.called_functions)
    function_recall = (
        len(expected_functions & called_functions) / len(expected_functions)
        if expected_functions
        else 1.0
    )
    fact_recall = sum(fact_checks.values()) / len(fact_checks) if fact_checks else 1.0

    return {
        "case_id": case.case_id,
        "split": case.split,
        "slice": case.slice,
        "architecture": run.architecture,
        "expected_placeholder": case.expected_placeholder,
        "actual_placeholder": parsed["placeholder"],
        "placeholder_syntax_valid": parsed["syntax_valid"],
        "placeholder_correct": parsed["placeholder"] == case.expected_placeholder,
        "required_fact_recall": fact_recall,
        "missing_fact_ids": [
            fact_id for fact_id, passed in fact_checks.items() if not passed
        ],
        "expected_functions": list(case.expected_functions),
        "called_functions": run.called_functions,
        "expected_function_recall": function_recall,
        "all_expected_functions_called": function_recall == 1.0,
        "tool_call_count": len(run.called_functions),
        "steps": run.steps,
        "prompt_tokens": run.prompt_tokens,
        "completion_tokens": run.completion_tokens,
        "total_tokens": run.total_tokens,
        "latency_seconds": run.latency_seconds,
        "final_answer": run.final_answer,
        "error": run.error,
        "execution_failed": run.error is not None,
    }

### Два агента на одном вопросе

Для проверки оценщика отправим один многошаговый вопрос в SGR и ReAct. Это smoke test, а не итоговый рейтинг.

Оба запуска проходят через одну функцию `evaluate_agent_run`. Таблица отдельно показывает маркер, обязательные факты и ожидаемые функции. Уже на одном кейсе можно увидеть, почему эти оси полезно хранить раздельно.


In [ ]:
case_m02_preview = EvalCase(
    "M02-preview",
    "multistep",
    "Верно ли, что TASK-106 назначена сотруднику команды, которой руководит Илья Морозов?",
    "<yes>",
    ("task_106_maria", "maria_agent_platform", "agent_platform_lead"),
    ("search_tasks", "find_employee", "get_team"),
    "связь задача -> сотрудник -> команда",
)

preview_runs = [
    run_sgr_agent(case_m02_preview.question),
    run_react_agent(case_m02_preview.question),
]

show_table(
    pd.DataFrame([evaluate_agent_run(case_m02_preview, run) for run in preview_runs])[
        [
            "architecture",
            "actual_placeholder",
            "placeholder_correct",
            "required_fact_recall",
            "expected_function_recall",
            "called_functions",
            "final_answer",
        ]
    ]
)

### Ограничение метрики по функциям

Посмотрите на строки, где `required_fact_recall` равен единице, а `expected_function_recall` ниже единицы. Такое возможно без ошибки в ответе.

Например, `get_team` уже возвращает список участников. Если агент нашёл команду и увидел в ней Марию Соколову, отдельный `find_employee` может не понадобиться. Текущая разметка всё равно посчитает одну ожидаемую функцию пропущенной.

Поэтому `expected_function_recall` здесь служит диагностикой маршрута. Для задач с несколькими допустимыми стратегиями важнее проверяемый результат и факты. В более строгом контуре понадобятся допустимые наборы маршрутов или проверки переходов состояния.

Ещё три вещи пока остаются вне автоматики:

- корректность аргументов функции;
- лишние или повторные вызовы;
- искажение результата после получения `count`.

Все данные для ручного разбора уже лежат в `trace`.


In [ ]:
def run_benchmark(
    cases: list[EvalCase],
    runner: Callable[[str], AgentRun],
    architecture: str,
) -> tuple[pd.DataFrame, dict[str, AgentRun]]:
    rows = []
    runs: dict[str, AgentRun] = {}
    for case in cases:
        started = time.perf_counter()
        try:
            run = runner(case.question)
        except Exception as error:
            run = AgentRun(
                architecture=architecture,
                final_answer="",
                called_functions=[],
                steps=0,
                prompt_tokens=0,
                completion_tokens=0,
                total_tokens=0,
                latency_seconds=time.perf_counter() - started,
                error=f"{type(error).__name__}: {error}",
            )
            print(f"{case.case_id}: {run.error}")
        runs[case.case_id] = run
        rows.append(evaluate_agent_run(case, run))
    return pd.DataFrame(rows), runs


def select_benchmark_cases(
    cases: list[EvalCase], preferred_ids: tuple[str, ...]
) -> list[EvalCase]:
    if FULL_BENCHMARK:
        return cases
    by_id = {case.case_id: case for case in cases}
    return [by_id[case_id] for case_id in preferred_ids if case_id in by_id]


def summarize_benchmark(frame: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame
    return frame.groupby("architecture", as_index=False).agg(
        cases=("case_id", "count"),
        failed_cases=("execution_failed", "sum"),
        error_rate=("execution_failed", "mean"),
        placeholder_accuracy=("placeholder_correct", "mean"),
        mean_required_fact_recall=("required_fact_recall", "mean"),
        mean_expected_function_recall=("expected_function_recall", "mean"),
        mean_tool_calls=("tool_call_count", "mean"),
        mean_steps=("steps", "mean"),
        mean_total_tokens=("total_tokens", "mean"),
        mean_latency_seconds=("latency_seconds", "mean"),
    )

### Первый прогон SGR на 18 простых кейсах

При ошибке API, разбора или контракта кейс не пропадает. Он получает `execution_failed=true`, текст исключения в `error` и нулевые метрики ответа. После этого запускается следующая строка.

В коротком режиме проверяются `T03` и `S03`: один старый вопрос по политикам и один новый вопрос к трекеру. При `LESSON8_FULL_BENCHMARK=1` выполняются все 18 кейсов.

Числа в таблице надо читать из вывода. Они зависят от реальных ответов модели и могут измениться при повторном запуске.


In [ ]:
sgr_18_cases = select_benchmark_cases(EVAL_18, ("T03", "S03"))
sgr_18_frame, sgr_18_runs = run_benchmark(sgr_18_cases, run_sgr_agent, "sgr_next_step")
show_table(summarize_benchmark(sgr_18_frame).round(3))
show_table(
    sgr_18_frame[
        [
            "case_id",
            "expected_placeholder",
            "actual_placeholder",
            "required_fact_recall",
            "called_functions",
            "steps",
            "total_tokens",
            "latency_seconds",
            "final_answer",
            "error",
        ]
    ]
)

### Как разбирать провал по траектории

Начинать лучше с первого неправильного перехода, а не с последнего предложения модели.

- Нужная функция не вызвана. Проверяем название, описание и пересечение с соседними инструментами.
- Функция выбрана, аргументы странные. Уточняем поля, определения статусов и примеры значений.
- Результат функции правильный, ответ ошибочный. Смотрим формат результата и инструкцию финального ответа.
- Одинаковый вызов повторяется. Проверяем, понял ли агент предыдущий результат и достаточно ли данных вернула функция.
- Шагов слишком много. Ищем инструмент, который заставляет собирать одну сущность по кусочкам.

Если список провалов пуст, ячейка просто сообщит об этом. Полезный разбор появится после полного прогона или намеренного ухудшения схемы в практике.


In [ ]:
failed_case_ids = sgr_18_frame.loc[
    (~sgr_18_frame["placeholder_correct"])
    | (sgr_18_frame["required_fact_recall"] < 1)
    | (sgr_18_frame["expected_function_recall"] < 1),
    "case_id",
].tolist()
print("Кейсы для разбора:", failed_case_ids)
if failed_case_ids:
    selected = sgr_18_runs[failed_case_ids[0]]
    show_table(pd.DataFrame(selected.trace))

## Добавляем пять многошаговых кейсов

Первые 18 задач в основном проверяют один источник. Они полезны для контрактов, но почти не нагружают агентный цикл.

В новых кейсах следующий объект поиска становится известен только после предыдущего результата.

| Кейс | Возможный маршрут | Ожидаемый результат |
|---|---|---|
| `M01` | команда -> задачи | `<count:4>` |
| `M02` | задача -> сотрудник -> команда | `<yes>` |
| `M03` | сотрудник -> команда -> задачи | `<count:2>` |
| `M04` | сотрудник + политика | `<no>` |
| `M05` | задача -> сотрудник + политика | `<no>` |

В колонке указан понятный маршрут, а не обязательная последовательность. Если другая траектория приводит к тем же проверяемым фактам, её нельзя автоматически считать ошибкой.


In [ ]:
MULTISTEP_CASES = [
    EvalCase(
        "M01",
        "multistep",
        "Сколько открытых задач по оценке агентов назначено участникам команды Agent Platform?",
        "<count:4>",
        ("support_eval_open_count", "agent_platform_size"),
        ("get_team", "search_tasks"),
        "команда -> задачи",
    ),
    EvalCase(
        "M02",
        "multistep",
        "Верно ли, что TASK-106 назначена сотруднику команды, которой руководит Илья Морозов?",
        "<yes>",
        ("task_106_maria", "maria_agent_platform", "agent_platform_lead"),
        ("search_tasks", "find_employee", "get_team"),
        "задача -> сотрудник -> команда",
    ),
    EvalCase(
        "M03",
        "multistep",
        "Сколько открытых задач с меткой prompt есть у команды, в которой работает Артем Волков?",
        "<count:2>",
        ("artem_agent_platform", "agent_platform_prompt_open_count"),
        ("find_employee", "get_team", "search_tasks"),
        "сотрудник -> команда -> задачи",
    ),
    EvalCase(
        "M04",
        "multistep",
        "Достаточно ли Анне Петровой согласования только HR для работы из другой страны?",
        "<no>",
        (
            "anna_agent_platform",
            "abroad_hr_approval",
            "abroad_legal_approval",
            "abroad_infosec_approval",
        ),
        ("find_employee", "search_policy_constraints"),
        "сотрудник + политика",
    ),
    EvalCase(
        "M05",
        "multistep",
        "Можно ли перенести шесть дней отпуска сотруднику, на которого назначена TASK-104, даже если руководитель письменно согласовал перенос?",
        "<no>",
        ("task_104_anna", "vacation_limit"),
        ("search_tasks", "find_employee", "search_policy_constraints"),
        "задача -> сотрудник + политика",
    ),
]

EVAL_23 = EVAL_18 + MULTISTEP_CASES
assert len(EVAL_23) == 23
assert len({case.case_id for case in EVAL_23}) == 23

### Неровный край в M01

`search_tasks` принимает одного исполнителя. Фильтра по списку сотрудников в текущем контракте нет. После получения команды агент может:

- запросить задачи проекта и проверить `team_id` исполнителей в результате;
- сделать отдельный поиск по нескольким участникам;
- воспользоваться совпадением состава команды и данных проекта в учебном наборе.

Оценка требует `get_team` и `search_tasks`, но не задаёт точную последовательность.

Этот кейс заодно показывает проблему дизайна инструмента. Если команда постоянно нужна как единица фильтрации, приложению может понадобиться отдельный параметр `team_id` или функция уровня `search_team_tasks`. Заставлять модель вручную изображать join удобно только до первого большого списка сотрудников.


## Сравниваем SGR и ReAct на общем наборе

Выше был один предварительный запуск `M02`. Он проверял оценщик и позволял открыть две траектории рядом.

Теперь запускаем общий эксперимент. Архитектуры получают одинаковые вопросы, функции и колонки оценки.

Короткий режим берёт три среза:

- `T03`, ответ по политике;
- `S03`, одна функция и точное число;
- `M02`, связь между тремя сущностями.

При `LESSON8_FULL_BENCHMARK=1` выполняются все 23 кейса. Число обращений к модели будет больше числа строк, потому что один кейс может занимать несколько шагов.

Один прогон остаётся демонстрацией. Для устойчивого вывода понадобятся повторы, доверительные интервалы и больше многошаговых задач.


In [ ]:
comparison_cases = select_benchmark_cases(EVAL_23, ("T03", "S03", "M02"))
sgr_23_frame, sgr_23_runs = run_benchmark(
    comparison_cases, run_sgr_agent, "sgr_next_step"
)
react_23_frame, react_23_runs = run_benchmark(
    comparison_cases, run_react_agent, "react_function_calling"
)
comparison_frame = pd.concat([sgr_23_frame, react_23_frame], ignore_index=True)

show_table(summarize_benchmark(comparison_frame).round(3))
show_table(
    comparison_frame[
        [
            "case_id",
            "architecture",
            "placeholder_correct",
            "required_fact_recall",
            "expected_function_recall",
            "tool_call_count",
            "steps",
            "total_tokens",
            "latency_seconds",
            "final_answer",
            "error",
        ]
    ].sort_values(["case_id", "architecture"])
)

In [ ]:
chart_data = summarize_benchmark(comparison_frame).set_index("architecture")
chart_data[["mean_steps", "mean_tool_calls"]].plot(kind="bar")
plt.title("Число шагов и вызовов функций")
plt.ylabel("Среднее на кейс")
plt.xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

### Как читать сравнение

Отдельной колонки `best_agent` здесь нет, и так было задумано.

- Высокая точность маркера при низкой полноте фактов означает, что агент угадал форму ответа и не обосновал её.
- Полный ответ при низком recall функций может быть допустимым альтернативным маршрутом или догадкой. Нужна траектория.
- Много вызовов при правильном результате показывает дорогую и, возможно, нестабильную траекторию.
- Низкие токены при ошибках означают только дешёвые ошибки.
- График шагов и функций ничего не говорит о качестве без соседней таблицы.

Для решения по архитектуре нужны повторные прогоны и более точная разметка аргументов. Текущий контур уже сохраняет данные, из которых такую оценку можно собрать.


### Решаем, что для нас значит «лучше»

Проанализируйте результаты и определите, какой агент лучше и, главное, почему.


## Добавляем планировщик

После длинной или повторяющейся траектории можно предпложить, что надо добавить планеровщик и будет все хорошо. Иногда он действительно помогает удерживать маршрут. Иногда добавляет ещё один вызов, который модель потом торжественно игнорирует.

В этом эксперименте план нужен как явное состояние приложения. Он может:

- пережить сжатие старой истории;
- быть виден человеку и системе наблюдаемости;
- участвовать в оценке повторов и прогресса.

Приложение назначает `update_plan` первым действием для многошагового запуска. Модель возвращает массив коротких проверяемых шагов, код сохраняет его и устанавливает `current_step=0`.

Это намеренное вмешательство в маршрут. При сравнении надо учитывать токены и дополнительный шаг планировщика.


In [ ]:
PLAN_STATE: dict[str, Any] = {
    "steps": [],
    "current_step": None,
    "status": "not_started",
}


def update_plan(steps: list[str]) -> dict[str, Any]:
    if not steps:
        raise ValueError("План должен содержать хотя бы один шаг")
    PLAN_STATE.update(
        {
            "steps": steps,
            "current_step": 0,
            "status": "in_progress",
        }
    )
    return copy.deepcopy(PLAN_STATE)


UPDATE_PLAN_SPEC = {
    "name": "update_plan",
    "description": (
        "Сохраняет короткий рабочий план в состоянии приложения. "
        "Используй для многошагового вопроса, когда нужно явно сохранить порядок действий. "
        "Не записывай внутренние рассуждения, только проверяемые действия."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "steps": {
                "type": "array",
                "items": {"type": "string"},
                "minItems": 1,
                "maxItems": 8,
                "description": "Короткие действия в порядке выполнения.",
            },
        },
        "required": ["steps"],
        "additionalProperties": False,
    },
}

plan_example = update_plan(["Найти сотрудника", "Получить команду", "Посчитать задачи"])
assert plan_example["current_step"] == 0
assert plan_example["status"] == "in_progress"
print(json.dumps(plan_example, ensure_ascii=False, indent=2))

### Подключаем план к ReAct

Для ReAct планировщик выглядит как ещё одна функция: добавляем спецификацию, Python-функция и правило не вызывать её повторно.

Первый вызов принудительно направляется в `update_plan`. После получения результата `tool_config` возвращается в режим `auto`, и модель снова сама выбирает инструменты.


In [ ]:
REACT_TOOLS_WITH_PLANNER = TOOL_SPECS + [UPDATE_PLAN_SPEC]
HANDLERS_WITH_PLANNER = {**TOOL_HANDLERS, "update_plan": update_plan}
REACT_PLANNER_SYSTEM_PROMPT = (
    REACT_SYSTEM_PROMPT
    + "\n\n"
    + """
6. Для многошагового вопроса первым действием вызови update_plan. Запиши в steps только проверяемые действия.
7. Если в истории уже есть результат update_plan, не вызывай его повторно.
""".strip()
)


def run_react_with_planner(question: str) -> AgentRun:
    return run_react_agent(
        question,
        tool_specs=REACT_TOOLS_WITH_PLANNER,
        system_prompt=REACT_PLANNER_SYSTEM_PROMPT,
        first_function_name="update_plan",
        handlers=HANDLERS_WITH_PLANNER,
    )


print("Функций без планировщика:", len(TOOL_SPECS))
print("Функций с планировщиком:", len(REACT_TOOLS_WITH_PLANNER))

### Подключаем план к SGR

В двухступенчатом SGR нужно изменить оба контракта. Селектор должен знать имя `update_plan`, а схема аргументов должна принять массив `steps`.

Если добавить только имя действия, модель сможет его выбрать, но приложение не поймёт, какую Pydantic-модель использовать дальше.

Поэтому ячейка одновременно:

- расширяет закрытый список действий;
- добавляет `UpdatePlanArguments`;
- строит итоговый `UpdatePlanAction`;
- назначает план первым шагом многошагового запуска.


In [ ]:
class UpdatePlanAction(BaseModel):
    model_config = ConfigDict(extra="forbid")
    name: Literal["update_plan"]
    steps: list[str] = Field(min_length=1, max_length=8)


class UpdatePlanArguments(BaseModel):
    model_config = ConfigDict(extra="forbid")
    steps: list[str] = Field(min_length=1, max_length=8)


class NextStepWithPlan(BaseModel):
    model_config = ConfigDict(extra="forbid")
    action_name: Literal[
        "search_policy_constraints",
        "find_employee",
        "get_team",
        "search_task_by_id_or_title",
        "filter_tasks_by_project",
        "filter_tasks_by_assignee",
        "update_plan",
        "final_answer",
    ]


def planner_argument_model_for_action(action_name: str) -> Type[BaseModel]:
    if action_name == "update_plan":
        return UpdatePlanArguments
    return argument_model_for_action(action_name)


def planner_argument_instruction_for_action(action_name: str) -> str:
    if action_name == "update_plan":
        return "Верни от 1 до 8 коротких проверяемых действий в steps. Не записывай внутренние рассуждения."
    return argument_instruction_for_action(action_name)


def build_action_with_planner(
    action_name: str,
    arguments: BaseModel,
) -> BaseModel:
    if action_name == "update_plan":
        return UpdatePlanAction.model_validate(
            {
                "name": "update_plan",
                **arguments.model_dump(mode="json"),
            }
        )
    return build_action_from_arguments(action_name, arguments)


SGR_PLANNER_SYSTEM_PROMPT = (
    SGR_SYSTEM_PROMPT
    + "\n\n"
    + """
Добавлено действие update_plan. Для многошагового вопроса приложение назначает его первым шагом.
После результата update_plan не выбирай его повторно. Массив steps приложение запросит отдельной схемой.
""".strip()
)


def run_sgr_with_planner(question: str) -> AgentRun:
    return run_sgr_agent(
        question,
        step_model=NextStepWithPlan,
        system_prompt=SGR_PLANNER_SYSTEM_PROMPT,
        argument_model_resolver=planner_argument_model_for_action,
        argument_instruction_resolver=planner_argument_instruction_for_action,
        action_builder=build_action_with_planner,
        initial_action_name="update_plan",
        handlers=HANDLERS_WITH_PLANNER,
    )


planner_choice_example = NextStepWithPlan(action_name="update_plan")
planner_arguments_example = UpdatePlanArguments(
    steps=["Найти задачу", "Найти сотрудника", "Проверить команду"],
)
planner_action_example = build_action_with_planner(
    planner_choice_example.action_name,
    planner_arguments_example,
)
assert isinstance(planner_action_example, UpdatePlanAction)

print(
    {
        "base_choice_schema_chars": len(
            json.dumps(NextStep.model_json_schema(), ensure_ascii=False)
        ),
        "planner_choice_schema_chars": len(
            json.dumps(NextStepWithPlan.model_json_schema(), ensure_ascii=False)
        ),
        "plan_arguments_schema_chars": len(
            json.dumps(UpdatePlanArguments.model_json_schema(), ensure_ascii=False)
        ),
    }
)

### Проверяем гипотезу о пользе плана

Обе архитектуры получают один и тот же многошаговый кейс. В полном режиме запускаются все пять `M`-кейсов.

Смотреть стоит на последствия, а не на красоту массива `steps`:

- изменилось ли качество ответа и обязательных фактов;
- стало ли меньше повторных функций;
- сколько шагов и токенов добавил план;
- появляется ли `update_plan` там, где хватило бы одного поиска.

Сам факт успешного вызова планировщика ещё ничего не улучшает. Польза должна проявиться в ответе или траектории.


In [ ]:
planner_cases = select_benchmark_cases(MULTISTEP_CASES, ("M02",))
sgr_planner_frame, _ = run_benchmark(
    planner_cases, run_sgr_with_planner, "sgr_next_step"
)
react_planner_frame, _ = run_benchmark(
    planner_cases, run_react_with_planner, "react_function_calling"
)
planner_comparison = pd.concat(
    [sgr_planner_frame, react_planner_frame], ignore_index=True
)
assert planner_comparison["error"].isna().all()
assert (
    planner_comparison["called_functions"]
    .apply(lambda names: "update_plan" in names)
    .all()
)
show_table(
    planner_comparison[
        [
            "case_id",
            "architecture",
            "placeholder_correct",
            "required_fact_recall",
            "expected_function_recall",
            "called_functions",
            "steps",
            "total_tokens",
            "error",
        ]
    ]
)

### Помог ли план в этом прогоне

Сравним одни и те же многошаговые кейсы с планировщиком и без него. Рост точности маркера и полноты фактов считаем пользой. Добавочные шаги и токены считаем ценой. `expected_function_recall` оставляем диагностикой: другой допустимый маршрут может снизить его без ошибки в ответе.


In [ ]:
planner_case_ids = [case.case_id for case in planner_cases]
baseline_rows = comparison_frame.loc[
    comparison_frame["case_id"].isin(planner_case_ids)
].assign(planner="without_plan")
planned_rows = planner_comparison.assign(planner="with_plan")
plan_effect_rows = pd.concat([baseline_rows, planned_rows], ignore_index=True)

plan_effect = plan_effect_rows.groupby(["architecture", "planner"], as_index=False).agg(
    cases=("case_id", "count"),
    placeholder_accuracy=("placeholder_correct", "mean"),
    mean_required_fact_recall=("required_fact_recall", "mean"),
    mean_expected_function_recall=("expected_function_recall", "mean"),
    mean_steps=("steps", "mean"),
    mean_total_tokens=("total_tokens", "mean"),
)
show_table(plan_effect.round(3))

for architecture in sorted(plan_effect["architecture"].unique()):
    current = plan_effect.loc[plan_effect["architecture"].eq(architecture)].set_index(
        "planner"
    )
    baseline = current.loc["without_plan"]
    planned = current.loc["with_plan"]
    marker_delta = planned["placeholder_accuracy"] - baseline["placeholder_accuracy"]
    fact_delta = (
        planned["mean_required_fact_recall"] - baseline["mean_required_fact_recall"]
    )
    step_delta = planned["mean_steps"] - baseline["mean_steps"]
    token_delta = planned["mean_total_tokens"] - baseline["mean_total_tokens"]

    if marker_delta > 0 or fact_delta > 0:
        verdict = "качество выросло; цену нужно сопоставить с этим ростом"
    elif marker_delta == 0 and fact_delta == 0 and (step_delta > 0 or token_delta > 0):
        verdict = "качество не изменилось, а шагов или токенов стало больше"
    elif marker_delta < 0 or fact_delta < 0:
        verdict = "качество снизилось"
    else:
        verdict = "по этим метрикам разницы нет"

    print(
        f"{architecture}: {verdict}. "
        f"Δшаги={step_delta:+.1f}, Δтокены={token_delta:+.0f}."
    )

print(
    "Один прогон на малом наборе показывает механику сравнения, а не устойчивый эффект планировщика."
)

## Контекст агента после нескольких шагов

Планировщик добавил ещё один объект состояния. К этому моменту агент уже работает не с простой историей чата, а с набором источников, из которых перед каждым вызовом собирается рабочий запрос:

```text
системная инструкция
+ описания функций или схема NextStep
+ вопрос пользователя
+ выбранные предыдущие вызовы и результаты
+ сохранённый план или другое состояние
+ контракт финального ответа
```

В статье Google [Architecting efficient context-aware multi-agent framework for production](https://developers.googleblog.com/architecting-efficient-context-aware-multi-agent-framework-for-production/) рабочий контекст описан как вычисляемое представление более богатого состояния сессии, памяти и артефактов. Для каждого вызова собирается новый срез.

В ноутбуке роли распределены так:

| Слой | Объект в коде |
|---|---|
| полная история запуска | `trace` |
| преобразование истории | `compact_native_messages` |
| запрос одного вызова | `request["messages"]` |
| явное состояние плана | `PLAN_STATE` |

Из старого поискового результата можно убрать длинные строки и дубли. Идентификаторы, числа, ошибки и последняя незавершённая пара вызов-результат должны сохраниться буквально.


### Рабочий контекст как представление состояния

Удобно мыслить не одним бесконечным массивом сообщений, а тремя слоями:

```text
Session / Trace
  хранит все события
        |
        v
Context builder
  выбирает инструкции, факты, последние пары и план
        |
        v
Working context
  уходит в один вызов модели
```

Полная траектория остаётся источником истины для аудита. Рабочий контекст можно пересобирать, сокращать и форматировать под конкретную модель.

Это разделение делает сбой проверяемым. `compact_native_messages` можно прогнать на фиксированной траектории, сравнить вход и выход и убедиться, что нужный `task_id` или `count` не исчез. Отлаживать абстрактное "модель забыла" заметно хуже.


## Как эта схема вырастает в универсального агента

Учебный агент знает четыре безопасные функции и отвечает на короткие вопросы. Кодинг-агент работает в той же базовой петле, но его среда гораздо шире: файлы, команды, права на изменение, тесты, долгие сессии и возврат к ошибке через десятки шагов.

В официальной статье OpenAI [Unrolling the Codex agent loop](https://openai.com/ru-RU/index/unrolling-the-codex-agent-loop/) приложение собирает запрос, принимает вызов инструмента, исполняет его и возвращает результат модели. Там же исполняющая обвязка называется `harness`.

Независимое исследование [Dive into Claude Code: The Design Space of Today's and Future AI Agent Systems](https://arxiv.org/abs/2604.14228) разбирает похожие инженерные слои в публично доступном коде: права, сжатие контекста, расширения, делегирование и хранение сессий. Это исследовательский анализ, а не официальная спецификация Anthropic.

| Слой обвязки | Ответственность |
|---|---|
| модель | выбрать следующий шаг и сформировать аргументы |
| исполнение функций | проверить и выполнить действие |
| состояние | хранить факты, план и прогресс |
| сборка контекста | выбрать рабочий срез для следующего вызова |
| ограничения | контролировать права, шаги и опасные операции |
| оценка | проверять результат, траекторию, стоимость и регрессии |
| человек | подтверждать рискованные действия и корректировать цель |

Чем универсальнее агент, тем большая доля надёжности переезжает в эту обвязку. Один удачный системный промпт её не заменяет.


In [ ]:
assert callable(dispatch_tool)
assert callable(compact_native_messages)
assert "trace" in AgentRun.__dataclass_fields__

## Практика

**Контракт инструмента**

Сделайте две плохие версии описания `search_tasks`: слишком общую и перегруженную деталями внутренней таблицы. Прогоните `S03`, `S07` и `S09`. Сравните аргументы, итоговый `count`, число повторов и токены.

**Выберите один вариант развития ReAct**

Сейчас `run_react_agent` сохраняет действие и результат. Выбер следующего шага в траектории не объяснён. Исправьте это одним из двух способов.

**Вариант A. Явное обоснование**

Перед выбором функции запрашивайте короткий структурированный ответ:

- `known_facts`: уже полученные проверяемые факты;
- `missing_fact`: какого факта не хватает для ответа;
- `next_action_reason`: как следующее действие добудет этот факт.

Приложение само записывает в `trace` вид события и номер шага. От модели не нужны внутренние идентификаторы и статусы.

**Вариант B. Штатное рассуждение и параллельное исполнение**

Включите [режим рассуждений GigaChat](https://developers.sber.ru/docs/ru/gigachat-b2bank/guides/reasoning) и сохраняйте в `trace` возвращённое сообщение с ролью `reasoning` или поле `reasoning_content`, в зависимости от формата API. Добавьте в приложение параллельное исполнение независимых безопасных функций чтения.

Параллелить можно только вызовы, которые не зависят от результатов друг друга. Например, два поиска по известным сущностям. Цепочка `find_employee -> get_team` остаётся последовательной, потому что `get_team` получает `team_id` из первого результата.

Для выбранного варианта прогоните `M02` и `M03` до и после изменения. Сравните факты в ответе, повторные вызовы, шаги, токены и полную задержку. Для варианта A дополнительно проверьте, совпало ли обоснование с фактическим следующим вызовом. Для варианта B покажите, какие вызовы действительно выполнялись одновременно.

**Опционально. Одна архитектура на разных моделях**

Оставьте код агента, инструменты, данные, набор задач и параметры запуска одинаковыми. Поменяйте только модель и прогоните одни и те же простые и многошаговые кейсы. Сравните итоговое качество и траекторию: какие функции выбраны, в каком порядке, где появились повторы, сколько понадобилось шагов, токенов и времени.

Главный вопрос эксперимента: точно ли более крупная модель даёт лучшее качество именно в этой архитектуре? Она может увереннее завершать сложные цепочки, а может платить за тот же результат лишними шагами и задержкой. Один удачный ответ ничего не решает, смотрите на весь фиксированный набор.

В качестве дополнительной модели можно попробовать [GigaChat 3 Ultra](https://developers.sber.ru/docs/ru/gigachat/models/gigachat-3-ultra). На момент подготовки занятия тариф для физлиц включает 50 миллионов бесплатных токенов для `GigaChat-3-Ultra` на 12 месяцев.


### Самопроверка

- Кто исполняет Python после ответа модели с `function_call`?
- Какие два сообщения добавляются в историю после функции?
- Почему `count` нельзя заменять длиной `items`?
- Чем SGR next-step отличается от workflow с маршрутом в коде?
- Какие ошибки скрывает простая проверка имени функции?
- Зачем хранить полную траекторию отдельно от рабочего контекста?
- Что сломается, если при сжатии оставить вызов без результата?
- По каким метрикам можно судить о пользе планировщика?
- Какие части универсального агента принадлежат harness, а не модели?


## Итоги

Мы начали с конфигурации `E`, которая умеет отвечать по заранее найденным политикам. Новый вопрос потребовал данных из справочника сотрудников, команд и трекера, поэтому в системе появились функции.

Затем вокруг одних функций были собраны две схемы управления. SGR возвращает структурированный следующий шаг и отдельно заполняет его аргументы. ReAct использует нативный function calling и повторяет цикл до текстового ответа.

Оценочный набор вырос до 23 кейсов: восемь регрессионных, десять простых инструментальных и пять многошаговых. Финальный ответ, факты, функции, шаги, токены и задержка проверяются отдельно.

Главная инженерная часть оказалась вокруг модели. Приложение исполняет функции, хранит траекторию, ограничивает цикл, собирает рабочий контекст и решает, когда нужен план. Чем длиннее задача, тем заметнее эта работа.


## Шпаргалка

- Новый источник данных сначала получает точечный инструмент и проверяемый контракт.
- Модель выбирает имя функции и аргументы. Доступы, исполнение и ошибки контролирует приложение.
- `function_call` и `function_result` хранятся согласованной парой.
- Старые RAG-кейсы остаются в регрессии после архитектурного изменения.
- Факты и итог важнее одного заранее придуманного маршрута.
- Полная траектория хранится целиком, рабочий контекст собирается для конкретного шага.
- Сжатие сохраняет идентификаторы, числа, ошибки и незавершённые действия.
- Планировщик считается полезным только после сравнения качества, повторов и дополнительных токенов.


## Полезные материалы

**GigaChat и вызовы функций**

- [GigaChat Python SDK](https://github.com/ai-forever/gigachat): модели запросов, `client.chat.parse`, `client.chat.create` и примеры function calling.
- [Режимы работы с функциями GigaChat](https://developers.sber.ru/docs/ru/gigachat/guides/functions/function-calling-modes): автоматический, отключённый и принудительный выбор.
- [Генерация аргументов пользовательских функций](https://developers.sber.ru/docs/ru/gigachat/guides/functions/generating-arguments-for-custom-functions): устройство запроса и ответа API.

**Архитектуры и инструменты**

- [Schema-Guided Reasoning: NextStep](https://abdullin.com/schema-guided-reasoning/demo): пример структурированного следующего шага.
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629): исходная работа про чередование действий и наблюдений.
- [Writing effective tools for AI agents](https://www.anthropic.com/engineering/writing-tools-for-agents): границы функций, схемы, ответы инструментов и оценка.
- [Don't Adapt Small Language Models for Tools; Adapt Tool Schemas to the Models](https://arxiv.org/abs/2510.07248): исследование влияния названий схем на tool calling небольших моделей.

**Контекст и harness**

- [Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents): отбор контекста, загрузка по необходимости и compaction.
- [Architecting efficient context-aware multi-agent framework for production](https://developers.googleblog.com/architecting-efficient-context-aware-multi-agent-framework-for-production/): рабочий контекст как вычисляемое представление состояния.
- [Unrolling the Codex agent loop](https://openai.com/ru-RU/index/unrolling-the-codex-agent-loop/): роль цикла, инструментов и исполняющей обвязки.
- [Dive into Claude Code: The Design Space of Today's and Future AI Agent Systems](https://arxiv.org/abs/2604.14228): независимый исследовательский разбор архитектуры кодинг-агента.
